<a href="https://colab.research.google.com/github/bogdanparvu18/msc-graduate-project/blob/main/notebooks/phase2/phase2_04_video_split_and_domain_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Sector 4: Video Splits and Domain-Adaptation References

This notebook reads the completed,
checksum-verified Sector 3 artifacts, splits videos with retained expert evidence,
propagates the split to annotations, assigns every physical video's role, and
builds a manifest of sampled unlabelled frame references for domain adaptation.

No video decodings, image extractions, GPU allocation, training, or pseudo-labelling
is performed. The last cell publishes only Sector 4 artifacts when `push=True`.

The input directory is located by `BOOTSTRAP`. Configuration is inherited from
the current Sector 3 snapshot; the new split/adaptation parameters below use the
original notebook's defaults. The inherited seed remains in force unless you
explicitly override it.


### 1. Dependencies and Imports

Only table processing and iterative multilabel stratification are needed.

In [ ]:
%pip install -q pandas numpy pyarrow iterative-stratification

In [ ]:
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path, PurePosixPath
import copy
import hashlib
import os
import re
import shutil
import tempfile
import subprocess
from urllib.parse import unquote, urlsplit
from numbers import Integral, Real
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from types import CodeType, FunctionType
import math
import importlib.metadata

### 2. Locate Sector 3 and declare the new Sector 4 parameters


In [ ]:
BOOTSTRAP = {
    "storage_root": "/content/drive/MyDrive/MMVQA_Clinical",
    "sector3_relative_dir": "outputs/phase2/sector3",
    "sector3_dir": None,
}

SECTOR4_SETTINGS = {
    "local_output_dir": "outputs/phase2/sector4",
    "export_dir": "outputs/phase2/sector4",
    "push": True,
    "dry_run": False,
    "github_overrides": {
    "token_secret_name": "GITHUB_TOKEN",
},
    "config_overrides": {
        "split_strategy": "video_level_multilabel_stratified",
        "train_fraction": 0.70,
        "validation_fraction": 0.15,
        "test_fraction": 0.15,
        "domain_adaptation_enabled": True,
        "domain_adaptation_include_fully_unlabelled_videos": True,
        "domain_adaptation_include_train_video_unlabelled_frames": True,
        "domain_adaptation_sampling_seconds": 0.5,
        # "seed": 42,  # Optional; otherwise inherited from Sector 3.
    },
}


### 3. Checked input loading and content-aware storage functions


In [ ]:
P3_STATE_PATH = "configs/sector3_state.json"
P3_RECEIPT_PATH = "configs/sector3_publication_receipt.json"
P3_NULL = "\\N"

S3_EXECUTION_METADATA_KEYS = frozenset({
    "run_id", "run_context", "run_datetime", "run_timestamp", "run_date", "run_time",
    "timestamp", "date", "time", "execution_date", "execution_time",
    "created_at", "created_at_utc", "created_utc", "saved_at", "saved_at_utc", "saved_utc",
    "started_at", "started_at_utc", "started_utc", "finished_at", "finished_at_utc", "finished_utc",
    "generated_at", "generated_at_utc", "generated_utc", "updated_at", "updated_at_utc", "updated_utc",
    "modified_at", "modified_at_utc", "modified_utc",
    "source_commit", "source_commit_sha", "source_commit_hash", "source_revision", "config_source_commit",
    "commit", "commit_sha", "head_commit", "current_commit", "repository_commit",
    "publication_receipt", "receipt", "last_publication",
    "source_configs", "config_provenance", "config_conflict_report", "lineage",
    "source_notebooks", "source_config_provenance", "config_sources", "literal_notebooks", "literal_configs",
    "input_sha256", "state_sha256", "upstream_states", "upstream_alignment_run_id",
    "notebook_path", "notebook_sha256", "source_notebook_path", "source_notebook_sha256",
    "source_config_path", "source_config_sha256", "source_snapshot_path", "source_snapshot_sha256",
    "config_snapshot_path", "config_snapshot_sha256", "snapshot_path", "snapshot_sha256",
    "source_config_file", "source_config_hash", "source_snapshot_file", "source_snapshot_hash",
    "config_snapshot_file", "config_snapshot_hash", "snapshot_file", "snapshot_hash",
    "config_inheritance_policy", "taxonomy_validation_policy",
})

def _s3_semantic_json_value(value):
    """Pure conversion to canonical JSON-compatible values; no I/O or clock access."""
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, dict):
        return {str(key): _s3_semantic_json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_s3_semantic_json_value(item) for item in value]
    if isinstance(value, (set, frozenset)):
        values = [_s3_semantic_json_value(item) for item in value]
        return sorted(values, key=lambda item: json.dumps(item, sort_keys=True))
    if isinstance(value, (Path, PurePosixPath)):
        return str(value)
    if isinstance(value, np.generic):
        return _s3_semantic_json_value(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        raise ValueError("Semantic configuration must not contain NaN or infinite values.")
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    raise TypeError(f"Unsupported semantic configuration value: {type(value).__name__}")

def _s3_semantic_json(value):
    return json.dumps(_s3_semantic_json_value(value), ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def _s3_semantic_digest(value):
    return hashlib.sha256(_s3_semantic_json(value).encode("utf-8")).hexdigest()

def sector3_semantic_config(config):
    """Project authored effective values, excluding execution/source bookkeeping only.

    Full source snapshots and current execution metadata remain available in the
    saved envelope. Authored paths, thresholds, publication settings (including
    push/dry_run), and unknown substantive config keys retain their meaning.
    """
    if not isinstance(config, dict):
        raise TypeError("Sector 3 semantic configuration requires a dictionary.")
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S3_EXECUTION_METADATA_KEYS}
        if isinstance(value, (list, tuple)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)

def sector3_semantic_config_fingerprint(config):
    """Stable identity of authored configuration, independent of execution time."""
    return _s3_semantic_digest(sector3_semantic_config(config))

def _p3_json_default(value):
    if value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, pd.DataFrame):
        return {"type": "dataframe", "columns": list(value.columns),
                "dtypes": {str(column): str(dtype) for column, dtype in value.dtypes.items()},
                "records": value.astype(object).where(value.notna(), None).to_dict("records")}
    if isinstance(value, (Path, PurePosixPath, datetime, pd.Timestamp)):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, (set, tuple)):
        return sorted(value) if isinstance(value, set) else list(value)
    raise TypeError(f"Unsupported Sector 3 JSON value: {type(value).__name__}")

def _p3_json_text(value):
    return json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                      allow_nan=False, default=_p3_json_default) + "\n"

def _p3_json_hash(value):
    return hashlib.sha256(_p3_json_text(value).encode("utf-8")).hexdigest()

def _p3_file_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

def _p3_relative(value):
    path = PurePosixPath(str(value))
    if path.is_absolute() or ".." in path.parts or len(path.parts) != 2:
        raise ValueError(f"Sector 3 artifacts must be flat relative paths: {value!r}")
    if path.parts[0] not in {"configs", "results", "reports"}:
        raise ValueError(f"Unexpected Sector 3 artifact group: {value!r}")
    return path.as_posix()

def _p3_safe_path(root, relative):
    path = root / _p3_relative(relative)
    if path.is_symlink() or root not in path.resolve().parents or path.parent.is_symlink():
        raise RuntimeError(f"Unsafe Sector 3 artifact path: {relative}")
    return path

def _p3_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(_p3_json_text(value), encoding="utf-8")
    if _p3_json_text(json.loads(path.read_text(encoding="utf-8"))) != _p3_json_text(value):
        raise RuntimeError(f"JSON readback failed: {path.name}")

def _p3_schema(frame):
    columns = []
    for column, dtype in frame.dtypes.items():
        descriptor = {"name": column, "dtype": str(dtype)}
        if isinstance(dtype, pd.StringDtype):
            descriptor["string_storage"] = dtype.storage
        if str(dtype) == "object":
            descriptor["object_kind"] = pd.api.types.infer_dtype(frame[column], skipna=True)
        if isinstance(dtype, pd.CategoricalDtype):
            descriptor.update({"categories": list(dtype.categories), "ordered": dtype.ordered})
        columns.append(descriptor)
    return {"rows": len(frame), "columns": columns, "column_axis_name": frame.columns.name,
            "csv_null_marker": P3_NULL, "schema_version": 1}

def sector3_table_fingerprint(frame):
    """Hash every value and its dtype schema; no clock or execution metadata is added."""
    if frame.columns.duplicated().any() or not all(isinstance(column, str) for column in frame.columns):
        raise ValueError("Sector 3 tables require unique string column names.")
    normalized = frame.reset_index(drop=True)
    digest = hashlib.sha256(_p3_json_text(_p3_schema(normalized)).encode("utf-8"))
    try:
        # Missing values have one semantic identity, including None/NaN/pd.NA in object columns.
        hash_values = normalized.astype(object).where(normalized.notna(), None)
        values = pd.util.hash_pandas_object(hash_values, index=False, categorize=False)
    except TypeError as error:
        raise TypeError("Sector 3 tables must contain scalar values; canonicalize nested conflict values first.") from error
    digest.update(values.to_numpy(dtype="uint64").astype("<u8", copy=False).tobytes())
    return digest.hexdigest()

def _p3_restore_csv(path, schema):
    frame = pd.read_csv(path, dtype="string", keep_default_na=False)
    expected = [descriptor["name"] for descriptor in schema["columns"]]
    if list(frame.columns) != expected or len(frame) != schema["rows"]:
        raise RuntimeError(f"CSV dimensions differ from saved schema: {path.name}")
    for descriptor in schema["columns"]:
        name, dtype = descriptor["name"], descriptor["dtype"]
        values = frame[name]
        nulls = values.eq(P3_NULL)
        values = values.mask(nulls, pd.NA)
        if dtype == "object":
            kind = descriptor.get("object_kind", "string")
            if kind in {"integer", "floating", "mixed-integer-float", "decimal"}:
                values = pd.to_numeric(values, errors="raise").astype(object)
            elif kind == "boolean":
                values = values.map({"True": True, "False": False}).astype(object)
            elif kind not in {"string", "unicode", "bytes", "empty"}:
                raise RuntimeError(f"Unsupported object CSV dtype {kind!r}: canonicalize this table before saving.")
            else:
                values = values.astype(object)
            frame[name] = values.mask(nulls, None)
        elif dtype in {"bool", "boolean"}:
            if (~nulls & ~values.isin(["True", "False"])).any():
                raise RuntimeError(f"Invalid saved boolean values: {path.name}/{name}")
            frame[name] = values.map({"True": True, "False": False}).astype(dtype)
        elif dtype == "string":
            frame[name] = values.astype(pd.StringDtype(storage=descriptor.get("string_storage", "python")))
        elif dtype == "category":
            category = pd.CategoricalDtype(descriptor["categories"], ordered=descriptor["ordered"])
            if pd.api.types.infer_dtype(descriptor["categories"], skipna=True) in {"integer", "floating", "mixed-integer-float"}:
                values = pd.to_numeric(values, errors="raise")
            frame[name] = values.astype(category)
        elif dtype.startswith("timedelta64"):
            frame[name] = pd.to_timedelta(values).astype(dtype)
        else:
            frame[name] = values.astype(dtype)
    frame.columns.name = schema.get("column_axis_name")
    return frame

def _p3_read_table(root, descriptor):
    path = _p3_safe_path(root, descriptor["relative_path"])
    schema_path = _p3_safe_path(root, descriptor["schema_path"])
    schema = json.loads(schema_path.read_text(encoding="utf-8"))
    if schema.get("fingerprint") != descriptor["fingerprint"] or schema.get("artifact") != descriptor["relative_path"]:
        raise RuntimeError("Table schema disagrees with the committed catalog.")
    frame = pd.read_parquet(path) if path.suffix == ".parquet" else _p3_restore_csv(path, schema)
    if sector3_table_fingerprint(frame) != descriptor["fingerprint"]:
        raise RuntimeError(f"Typed table readback fingerprint failed: {path.name}")
    return frame

def _p3_catalog_meaning(state):
    return {key: state[key] for key in (
        "schema_version", "complete", "processing_signature", "semantic_config_fingerprint",
        "input_sha256", "core_source_sha256", "config_snapshot", "tables", "summary", "artifacts", "obsolete_paths")}

def _p3_read_state(root):
    state_path = _p3_safe_path(root, P3_STATE_PATH)
    if not state_path.exists():
        return None
    try:
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if state.get("schema_version") != 2 or state.get("complete") is not True:
            raise RuntimeError("Sector 3 catalog is incomplete or uses an obsolete layout.")
        if state.get("content_id") != _p3_json_hash(_p3_catalog_meaning(state)):
            raise RuntimeError("Sector 3 catalog content fingerprint is invalid.")
        records = {record["relative_path"]: record for record in state["artifacts"]}
        if len(records) != len(state["artifacts"]) or P3_STATE_PATH in records or P3_RECEIPT_PATH in records:
            raise RuntimeError("Sector 3 catalog contains duplicate or operational artifact entries.")
        for relative, record in records.items():
            path = _p3_safe_path(root, relative)
            if not path.is_file() or path.stat().st_size != record["bytes"] or _p3_file_hash(path) != record["sha256"]:
                raise RuntimeError(f"Sector 3 artifact checksum verification failed: {relative}")
        if state["config_snapshot"] not in records:
            raise RuntimeError("Sector 3 config snapshot is absent from its catalog.")
        config_envelope = json.loads((root / state["config_snapshot"]).read_text(encoding="utf-8"))
        if (config_envelope.get("semantic_config_fingerprint") != state["semantic_config_fingerprint"]
                or sector3_semantic_config_fingerprint(config_envelope["effective_config"]) != state["semantic_config_fingerprint"]):
            raise RuntimeError("Sector 3 configuration snapshot disagrees with its catalog.")
        for field in ("summary_relative_path", "report_relative_path"):
            if state["summary"][field] not in records:
                raise RuntimeError("Sector 3 summary or executive report is absent from its catalog.")
        summary = json.loads((root / state["summary"]["summary_relative_path"]).read_text(encoding="utf-8"))
        if _p3_json_hash(summary) != state["summary"]["fingerprint"]:
            raise RuntimeError("Sector 3 summary disagrees with its catalog.")
        if records[state["summary"]["report_relative_path"]]["sha256"] != state["summary"]["report_fingerprint"]:
            raise RuntimeError("Sector 3 executive report disagrees with its catalog.")
        for descriptor in state["tables"].values():
            if descriptor["relative_path"] not in records or descriptor["schema_path"] not in records:
                raise RuntimeError("Sector 3 table or schema is absent from its catalog.")
            schema = json.loads((root / descriptor["schema_path"]).read_text(encoding="utf-8"))
            if schema.get("artifact") != descriptor["relative_path"] or schema.get("fingerprint") != descriptor["fingerprint"]:
                raise RuntimeError("Sector 3 saved table schema disagrees with the catalog.")
        for obsolete in state["obsolete_paths"]:
            if PurePosixPath(_p3_relative(obsolete)).parts[0] != "results" or obsolete in records:
                raise RuntimeError("Invalid obsolete-result deletion authorization.")
        return state
    except (OSError, ValueError, KeyError, TypeError) as error:
        raise RuntimeError(f"Cannot verify the existing Sector 3 catalog: {error}") from error

S4_STATE_PATH = 'configs/sector4_state.json'
S4_INPUT_NAMES = ('labelled_frame_manifest', 'video_frame_manifest', 'video_class_matrix', 'video_class_support')
S4_RESULT_NAMES = frozenset({'labelled_frame_manifest_with_split', 'video_manifest_with_roles',
                              'domain_source_videos', 'domain_adaptation_manifest', 'video_split_manifest'})
S4_EXECUTION_METADATA_KEYS = frozenset({
    'run_id', 'run_context', 'run_datetime', 'run_timestamp', 'run_date', 'run_time',
    'timestamp', 'date', 'time', 'execution_date', 'execution_time',
    'created_at', 'created_at_utc', 'created_utc', 'saved_at', 'saved_at_utc', 'saved_utc',
    'started_at', 'started_at_utc', 'started_utc', 'finished_at', 'finished_at_utc', 'finished_utc',
    'generated_at', 'generated_at_utc', 'generated_utc', 'updated_at', 'updated_at_utc', 'updated_utc',
    'modified_at', 'modified_at_utc', 'modified_utc',
})
S4_FRESH_CONFIG_KEYS = frozenset({
    'train_fraction', 'validation_fraction', 'test_fraction', 'split_strategy',
    'domain_adaptation_enabled', 'domain_adaptation_include_fully_unlabelled_videos',
    'domain_adaptation_include_train_video_unlabelled_frames', 'domain_adaptation_sampling_seconds',
})


def _p4_assert_no_secrets(value, path='configuration'):
    """Allow secret references, but never persist credential material."""
    if isinstance(value, dict):
        for key, item in value.items():
            label = str(key).lower()
            if label in {'token', 'access_token', 'github_token', 'password', 'api_key',
                         'authorization', 'secret', 'private_key', 'client_secret'} and item:
                raise ValueError(f'Credential material cannot be saved at {path}.{key}. Use token_secret_name.')
            _p4_assert_no_secrets(item, f'{path}.{key}')
    elif isinstance(value, (list, tuple)):
        for item in value:
            _p4_assert_no_secrets(item, path)
    elif isinstance(value, str):
        if re.search(r'https?://[^/\s]*:[^/@\s]+@', value) or re.search(r'(?:gh[pousr]_[A-Za-z0-9_]{20,}|github_pat_[A-Za-z0-9_]{20,})', value):
            raise ValueError(f'Credential material cannot be saved at {path}.')


def _p4_merge(base, overrides):
    result = copy.deepcopy(base)
    for key, value in overrides.items():
        result[key] = _p4_merge(result[key], value) if (
            isinstance(result.get(key), dict) and isinstance(value, dict)) else copy.deepcopy(value)
    return result


def _p4_config_report(source, effective, prefix=''):
    records = []
    for key in sorted(set(source) | set(effective)):
        if str(key).lower() in S4_EXECUTION_METADATA_KEYS:
            continue
        name = f'{prefix}.{key}' if prefix else str(key)
        left, right = source.get(key, '<not declared>'), effective.get(key, '<not declared>')
        if isinstance(left, dict) and isinstance(right, dict):
            records.extend(_p4_config_report(left, right, name))
        else:
            records.append({'config_key': name, 'sector3_value': _s3_semantic_json(left),
                            'sector4_value': _s3_semantic_json(right),
                            'resolution': 'inherited' if left == right else 'sector4_override'})
    return records


def sector4_semantic_config(config):
    """Exclude clock bookkeeping only from the effective authored Sector 4 config."""
    def project(value):
        if isinstance(value, dict):
            return {str(key): project(item) for key, item in value.items()
                    if str(key).lower() not in S4_EXECUTION_METADATA_KEYS}
        if isinstance(value, (tuple, list)):
            return [project(item) for item in value]
        return _s3_semantic_json_value(value)
    return project(config)


def sector4_semantic_config_fingerprint(config):
    return _s3_semantic_digest(sector4_semantic_config(config))


def io_load_sector4_context(bootstrap, settings):
    """Consume the current, checked Sector 3 state and its effective config snapshot.

    Split/domain defaults must be authored in Sector 4 settings/config_overrides.
    Stale envelope lineage and historical Sector 1/2 snapshots are never input truth.
    """
    if not isinstance(bootstrap, dict) or not isinstance(settings, dict):
        raise TypeError('BOOTSTRAP and SECTOR4_SETTINGS must be dictionaries.')
    storage_root = Path(bootstrap.get('storage_root') or '/content/drive/MyDrive/MMVQA_Clinical').expanduser().resolve()
    declared = bootstrap.get('sector3_dir')
    source_root = Path(declared).expanduser().resolve() if declared else (
        storage_root / bootstrap.get('sector3_relative_dir', 'outputs/phase2/sector3')).resolve()
    state = _p3_read_state(source_root)
    if state is None:
        raise FileNotFoundError(f'Run Sector 3 first: missing {source_root / P3_STATE_PATH}')
    missing = sorted(set(S4_INPUT_NAMES) - set(state['tables']))
    if missing:
        raise KeyError(f'The completed Sector 3 catalog lacks required tables: {missing}')
    inputs, consumed_hashes, consumed_descriptors = {}, {}, {}
    for name in S4_INPUT_NAMES:
        descriptor = state['tables'][name]
        inputs[name] = _p3_read_table(source_root, descriptor)
        consumed_descriptors[name] = copy.deepcopy(descriptor)
        for field in ('relative_path', 'schema_path'):
            relative = descriptor[field]
            consumed_hashes[f'{name}/{field}'] = _p3_file_hash(_p3_safe_path(source_root, relative))
    snapshot_path = _p3_safe_path(source_root, state['config_snapshot'])
    envelope = json.loads(snapshot_path.read_text(encoding='utf-8'))
    current_source = envelope['effective_config']
    # Checked persisted bytes remain the provenance; only live access paths are rebased.
    previous_storage = Path(current_source['storage_root']).expanduser().resolve()
    if previous_storage != storage_root:
        for name, table in inputs.items():
            for path_column, relative_column in (('image_path', 'image_relpath'), ('video_path', 'video_relpath')):
                if path_column not in table:
                    continue
                if relative_column not in table:
                    raise KeyError(f'Cannot rebase live {name}.{path_column} without {relative_column}.')
                rebased = []
                for value in table[relative_column]:
                    if pd.isna(value):
                        rebased.append(pd.NA)
                        continue
                    relative = PurePosixPath(str(value))
                    if relative.is_absolute() or '..' in relative.parts or '\\' in str(value):
                        raise ValueError(f'Unsafe portable path in {name}.{relative_column}.')
                    rebased.append(str(storage_root.joinpath(*relative.parts)))
                table[path_column] = pd.Series(rebased, index=table.index).astype(table[path_column].dtype)
    # Strip historical provenance blocks; keep all authored effective source values.
    source = sector3_semantic_config(current_source)
    overrides = copy.deepcopy(settings.get('config_overrides', {}))
    for key in S4_FRESH_CONFIG_KEYS | {'seed'}:
        if key in settings:
            overrides[key] = copy.deepcopy(settings[key])
    missing_fresh = sorted(S4_FRESH_CONFIG_KEYS - set(overrides))
    if missing_fresh:
        raise KeyError(f'Sector 4 must declare its own split/domain settings, missing: {missing_fresh}')
    inherited = {key: value for key, value in source.items() if key not in S4_FRESH_CONFIG_KEYS}
    config = _p4_merge(inherited, overrides)
    local_output_dir = settings.get('local_output_dir', 'outputs/phase2/sector4')
    export_dir = settings.get('export_dir', 'outputs/phase2/sector4')
    if export_dir != 'outputs/phase2/sector4':
        raise ValueError('Sector 4 export_dir must be outputs/phase2/sector4.')
    sector_settings = {key: copy.deepcopy(value) for key, value in settings.items()
                       if key not in S4_FRESH_CONFIG_KEYS | {'seed', 'config_overrides', 'github_overrides'}}
    sector_settings.update({'local_output_dir': local_output_dir, 'export_dir': export_dir})
    github = copy.deepcopy(source.get('github', {}))
    # Preserve the inherited repository/account contract, with publication confined to Sector 4.
    github_overrides = copy.deepcopy(settings.get('github_overrides', {}))
    for key in ('repo_url', 'branch', 'token_secret_name'):
        if key in github_overrides and github_overrides[key] != github.get(key):
            raise ValueError(f'Sector 4 must inherit the current Sector 3 GitHub {key}.')
    github.update(github_overrides)
    github.update({'repo_output_dir': export_dir, 'push': bool(settings.get('push', False)),
                   'dry_run': bool(settings.get('dry_run', False)),
                   'commit_message': 'Phase 2 Sector 4: publish video splits and domain-adaptation manifests'})
    github.pop('confirm_push', None)
    config.update({'phase': 2, 'sector': 4, 'storage_root': str(storage_root),
                   'sector4': sector_settings, 'github': github})
    _p4_assert_no_secrets(config)
    report = pd.DataFrame.from_records(_p4_config_report(source, config),
        columns=['config_key', 'sector3_value', 'sector4_value', 'resolution']).astype('string')
    lineage = {
        'sector3_root': str(source_root),
        'sector3_state_path': str(source_root / P3_STATE_PATH),
        'sector3_state_sha256': _p3_file_hash(source_root / P3_STATE_PATH),
        'sector3_content_id': state['content_id'],
        'sector3_config_snapshot': state['config_snapshot'],
        'sector3_config_snapshot_sha256': _p3_file_hash(snapshot_path),
        'sector3_semantic_config_fingerprint': state['semantic_config_fingerprint'],
        'input_sha256': consumed_hashes,
        'input_descriptors': consumed_descriptors,
        'config_conflict_report': report,
        'config_inheritance_policy': 'Current Sector 3 effective config; explicit Sector 4 split/domain defaults and overrides.',
    }
    return config, inputs, lineage


def _p4_root(config):
    output = Path(config.get('sector4', {}).get('local_output_dir', 'outputs/phase2/sector4'))
    return (output if output.is_absolute() else Path(config['storage_root']) / output).resolve()


def _p4_catalog_meaning(state):
    return _p3_catalog_meaning(state)


def _p4_read_state(root):
    """Check all committed bytes, schemas, current config, and semantic summary."""
    root = Path(root).resolve()
    state_path = _p3_safe_path(root, S4_STATE_PATH)
    if not state_path.exists():
        return None
    state = json.loads(state_path.read_text(encoding='utf-8'))
    if state.get('schema_version') != 2 or state.get('complete') is not True:
        raise RuntimeError('Sector 4 catalog is incomplete or has an unsupported schema.')
    if state.get('content_id') != _p3_json_hash(_p4_catalog_meaning(state)):
        raise RuntimeError('Sector 4 catalog fingerprint is invalid.')
    records = {record['relative_path']: record for record in state['artifacts']}
    if len(records) != len(state['artifacts']) or S4_STATE_PATH in records:
        raise RuntimeError('Sector 4 catalog contains duplicate or operational entries.')
    for relative, record in records.items():
        path = _p3_safe_path(root, relative)
        if not path.is_file() or path.stat().st_size != record['bytes'] or _p3_file_hash(path) != record['sha256']:
            raise RuntimeError(f'Sector 4 artifact checksum verification failed: {relative}')
    if state['config_snapshot'] not in records:
        raise RuntimeError('Sector 4 current config snapshot is absent from catalog.')
    envelope = json.loads(_p3_safe_path(root, state['config_snapshot']).read_text(encoding='utf-8'))
    if envelope.get('semantic_config_fingerprint') != state['semantic_config_fingerprint'] or (
        sector4_semantic_config_fingerprint(envelope['effective_config']) != state['semantic_config_fingerprint']):
        raise RuntimeError('Sector 4 effective config snapshot disagrees with its catalog.')
    for name, descriptor in state['tables'].items():
        if descriptor['relative_path'] not in records or descriptor['schema_path'] not in records:
            raise RuntimeError(f'Sector 4 table/schema absent from catalog: {name}')
        schema = json.loads(_p3_safe_path(root, descriptor['schema_path']).read_text(encoding='utf-8'))
        if schema.get('artifact') != descriptor['relative_path'] or schema.get('fingerprint') != descriptor['fingerprint']:
            raise RuntimeError(f'Sector 4 table schema disagrees with catalog: {name}')
    summary = state['summary']
    if summary['summary_relative_path'] not in records or summary['report_relative_path'] not in records:
        raise RuntimeError('Sector 4 summary/report absent from catalog.')
    saved_summary = json.loads(_p3_safe_path(root, summary['summary_relative_path']).read_text(encoding='utf-8'))
    if _p3_json_hash(saved_summary) != summary['fingerprint'] or records[summary['report_relative_path']]['sha256'] != summary['report_fingerprint']:
        raise RuntimeError('Sector 4 summary/report disagrees with catalog.')
    for relative in state.get('obsolete_paths', []):
        if not _p3_relative(relative).startswith('results/') or relative in records:
            raise RuntimeError('Invalid Sector 4 obsolete-result deletion authorization.')
    return state


def io_load_cached_sector4(config, processing_signature):
    state = _p4_read_state(_p4_root(config))
    if state is None or state['processing_signature'] != processing_signature:
        return None
    tables = {name: _p3_read_table(_p4_root(config), descriptor)
              for name, descriptor in state['tables'].items()}
    return {'tables': tables, 'state': state}


def _p4_summary(tables):
    summary = {'phase': 2, 'sector': 4,
               'table_row_counts': {name: len(table) for name, table in sorted(tables.items())},
               'units': {'video_split_manifest': 'video', 'labelled_frame_manifest_with_split': 'accepted annotation',
                         'domain_adaptation_manifest': 'unlabelled frame reference'},
               'policy': 'Validation/test videos never contribute domain-adaptation frames. References do not imply clinical normality.'}
    for name in ('video_split_manifest', 'video_role_report', 'domain_adaptation_summary'):
        if name in tables:
            summary[name] = tables[name].astype(object).where(tables[name].notna(), None).to_dict('records')
    return summary


def _p4_executive(summary):
    counts = summary['table_row_counts']
    return ('# Phase 2 — Sector 4: Video Splits and Domain Adaptation\n\n'
            'This sector partitions accepted labelled videos into reproducible train, validation, and test sets, '
            'propagates those assignments to accepted annotations, and assigns audited roles to every source video.\n\n'
            f"The split manifest contains **{counts.get('video_split_manifest', 0):,} labelled videos**. "
            f"The domain-adaptation manifest contains **{counts.get('domain_adaptation_manifest', 0):,} unlabelled frame references**.\n\n"
            'Validation and test videos are excluded from adaptation. Train videos with rejected source annotations '
            'are blocked from adaptation; accepted labelled frame indices are excluded from the sampled pool. '
            'Unlabelled frames are not assumed to be clinically normal. No frames are decoded or exported here.\n\n'
            'Results have stable filenames. Config snapshots and report versions are created only when their content changes.\n')


def _p4_write_table(stage, name, frame, fingerprint, stamp):
    result = name in S4_RESULT_NAMES
    base = f'results/{name}' if result else f'reports/{name}_{stamp}'
    relative = base + ('.csv' if name == 'video_split_manifest' or not result else '.parquet')
    path = stage / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if relative.endswith('.parquet'):
        frame.to_parquet(path, index=False, compression='zstd')
        restored = pd.read_parquet(path)
    else:
        serialized = frame.astype('string')
        if serialized.eq(P3_NULL).fillna(False).any().any():
            raise ValueError(f'Reserved CSV null sentinel occurs in {name}.')
        serialized.fillna(P3_NULL).to_csv(path, index=False)
        restored = _p3_restore_csv(path, _p3_schema(frame))
    pd.testing.assert_frame_equal(frame.reset_index(drop=True), restored, check_exact=True, check_dtype=True)
    schema_relative = base + '_schema.json'
    _p3_write_json(stage / schema_relative,
                  {**_p3_schema(frame), 'artifact': relative, 'fingerprint': fingerprint})
    return {'relative_path': relative, 'schema_path': schema_relative, 'fingerprint': fingerprint}


def _p4_install(stage, root, changed_paths, obsolete_paths):
    """Rollback on errors; atomic current catalog replacement occurs last."""
    if not root.exists():
        stage.rename(root)
        return
    backup = Path(tempfile.mkdtemp(prefix='.sector4-backup-', dir=root.parent))
    installed, moved = [], []
    sequence = [value for value in changed_paths if value != S4_STATE_PATH] + list(obsolete_paths) + [S4_STATE_PATH]
    try:
        for relative in dict.fromkeys(sequence):
            target = _p3_safe_path(root, relative)
            if target.exists():
                saved = backup / relative
                saved.parent.mkdir(parents=True, exist_ok=True)
                target.replace(saved)
                moved.append(relative)
            source = stage / relative
            if source.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                source.replace(target)
                installed.append(relative)
    except BaseException:
        for relative in reversed(installed):
            (root / relative).unlink(missing_ok=True)
        for relative in moved:
            target = root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            (backup / relative).replace(target)
        raise
    finally:
        shutil.rmtree(backup, ignore_errors=True)


def io_persist_sector4(tables, config, lineage, signature):
    """Do not touch any saved file when config, provenance and output content agree."""
    if not isinstance(signature, str) or not re.fullmatch(r'[0-9a-f]{64}', signature):
        raise ValueError('Pass the explicit Sector 4 SHA-256 processing signature.')
    if not tables or any(not isinstance(frame, pd.DataFrame) for frame in tables.values()):
        raise TypeError('Sector 4 output tables must be a nonempty dictionary of DataFrames.')
    if any(not re.fullmatch(r'[a-z][a-z0-9_]*', name) for name in tables):
        raise ValueError('Unsafe Sector 4 output table name.')
    _p4_assert_no_secrets(config)
    _p4_assert_no_secrets(lineage)
    root = _p4_root(config)
    previous = _p4_read_state(root)
    config_hash = sector4_semantic_config_fingerprint(config)
    provenance = {'input_sha256': copy.deepcopy(lineage['input_sha256']),
                  'core_source_sha256': lineage.get('core_source_sha256')}
    for digest in provenance['input_sha256'].values():
        if not isinstance(digest, str) or not re.fullmatch(r'[0-9a-f]{64}', digest):
            raise ValueError('Invalid consumed input SHA-256 in Sector 4 lineage.')
    core_hash = provenance['core_source_sha256']
    if core_hash is not None and not re.fullmatch(r'[0-9a-f]{64}', core_hash):
        raise ValueError('Invalid Sector 4 core-source SHA-256.')
    fingerprints = {name: sector3_table_fingerprint(table) for name, table in tables.items()}
    summary = _p4_summary(tables)
    summary_hash = _p3_json_hash(summary)
    current_report_hash = hashlib.sha256(_p4_executive(summary).encode('utf-8')).hexdigest()
    same_tables = previous is not None and set(fingerprints) == set(previous['tables']) and all(
        previous['tables'][name]['fingerprint'] == fingerprint for name, fingerprint in fingerprints.items())
    if previous and same_tables and all(previous[key] == value for key, value in provenance.items()) and (
        previous['semantic_config_fingerprint'] == config_hash and previous['summary']['fingerprint'] == summary_hash
        and previous['processing_signature'] == signature
        and previous['summary']['report_fingerprint'] == current_report_hash):
        # Readback detects a typed table damaged without byte/catalog inconsistency.
        for descriptor in previous['tables'].values():
            _p3_read_table(root, descriptor)
        return {'status': 'unchanged', 'root': str(root), 'local_root': str(root),
                'state': previous, 'artifacts': previous['artifacts'], 'changed_paths': []}
    root.parent.mkdir(parents=True, exist_ok=True)
    stage = Path(tempfile.mkdtemp(prefix='.sector4-stage-', dir=root.parent))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    try:
        catalog = {record['relative_path']: record for record in previous['artifacts']} if previous else {}
        descriptors = {}
        obsolete = set(previous.get('obsolete_paths', [])) if previous else set()
        config_snapshot = previous['config_snapshot'] if previous else None
        if not previous or previous['semantic_config_fingerprint'] != config_hash:
            config_snapshot = f'configs/sector4_config_{stamp}.json'
            _p3_write_json(stage / config_snapshot, {'schema_version': 2, 'changed_at_utc': stamp,
                'semantic_config_fingerprint': config_hash, 'effective_config': config, 'lineage': lineage})
        for name, frame in sorted(tables.items()):
            old = previous['tables'].get(name) if previous else None
            if old and old['fingerprint'] == fingerprints[name]:
                # Verified current bytes before reusing descriptors.
                _p3_read_table(root, old)
                descriptors[name] = old
            else:
                descriptors[name] = _p4_write_table(stage, name, frame, fingerprints[name], stamp)
                if old and old['relative_path'].startswith('results/') and old['relative_path'] != descriptors[name]['relative_path']:
                    for relative in (old['relative_path'], old['schema_path']):
                        obsolete.add(relative)
                        catalog.pop(relative, None)
        if previous:
            for name, old in previous['tables'].items():
                if name not in descriptors and old['relative_path'].startswith('results/'):
                    for relative in (old['relative_path'], old['schema_path']):
                        obsolete.add(relative)
                        catalog.pop(relative, None)
        previous_summary = previous['summary'] if previous else {}
        summary_path = previous_summary.get('summary_relative_path')
        if previous_summary.get('fingerprint') != summary_hash:
            summary_path = f'reports/summary_{stamp}.json'
            _p3_write_json(stage / summary_path, summary)
        report_text = _p4_executive(summary)
        report_hash = hashlib.sha256(report_text.encode('utf-8')).hexdigest()
        report_path = previous_summary.get('report_relative_path')
        if catalog.get(report_path, {}).get('sha256') != report_hash:
            report_path = f'reports/executive_report_{stamp}.md'
            (stage / report_path).write_text(report_text, encoding='utf-8')
        summary_descriptor = {'fingerprint': summary_hash, 'summary_relative_path': summary_path,
                              'report_fingerprint': report_hash, 'report_relative_path': report_path}
        changed = []
        for path in sorted(stage.rglob('*')):
            if path.is_file():
                relative = path.relative_to(stage).as_posix()
                record = {'relative_path': relative, 'sha256': _p3_file_hash(path), 'bytes': path.stat().st_size}
                if catalog.get(relative) == record:
                    path.unlink()
                else:
                    catalog[relative] = record
                    changed.append(relative)
        obsolete.difference_update(catalog)
        state = {'schema_version': 2, 'complete': True, 'processing_signature': signature,
                 'semantic_config_fingerprint': config_hash, 'config_snapshot': config_snapshot,
                 **provenance, 'tables': descriptors, 'summary': summary_descriptor,
                 'artifacts': [catalog[path] for path in sorted(catalog)], 'obsolete_paths': sorted(obsolete)}
        state['content_id'] = _p3_json_hash(_p4_catalog_meaning(state))
        state['changed_at_utc'] = stamp
        _p3_write_json(stage / S4_STATE_PATH, state)
        changed.append(S4_STATE_PATH)
        _p4_install(stage, root, changed, sorted(obsolete))
        # State is installed last; verify that every committed artifact is readable.
        _p4_read_state(root)
        return {'status': 'updated', 'root': str(root), 'local_root': str(root),
                'state': state, 'artifacts': state['artifacts'], 'changed_paths': changed}
    finally:
        shutil.rmtree(stage, ignore_errors=True)


def io_publish_sector4(persisted, config):
    """Optionally export a verified Sector 4 snapshot using an isolated Git clone.

    Call only after local persistence succeeds. No existing checkout is changed.
    Requires the existing _p4_read_state, _p3_safe_path and _p3_file_hash helpers.
    Tokens are supplied only to child processes through a private askpass helper.
    """

    github = config.get("github", {})
    sector = config.get("sector4", {})
    if not isinstance(github, dict) or not isinstance(sector, dict):
        raise TypeError("github and sector4 configuration must be dictionaries.")
    root_value = persisted.get("local_root", persisted.get("root"))
    if root_value is None:
        raise ValueError("The persisted Sector 4 root is missing.")
    root = Path(root_value).resolve()
    state = _p4_read_state(root)
    if state is None:
        raise RuntimeError("Sector 4 has no valid persisted snapshot to publish.")
    if persisted.get("state") != state:
        raise RuntimeError("Sector 4 changed after persistence; persist it again before publishing.")

    def strict_bool(value, name):
        if not isinstance(value, bool):
            raise TypeError(f"{name} must be a JSON/Python boolean.")
        return value

    dry_run = strict_bool(sector.get("dry_run", github.get("dry_run", False)), "dry_run")
    push = strict_bool(sector.get("push", github.get("push", True)), "push")
    allow_local = strict_bool(
        github.get("allow_local_test_remote", sector.get("allow_local_test_remote", False)),
        "allow_local_test_remote",
    )
    export_dir = str(sector.get("export_dir", "outputs/phase2/sector4"))
    export_parts = PurePosixPath(export_dir)
    if (
        export_parts.is_absolute()
        or not export_parts.parts
        or any(part in {".", "..", ".git"} for part in export_parts.parts)
        or "\\" in export_dir
        or "\x00" in export_dir
        or export_dir != export_parts.as_posix()
        or export_dir != 'outputs/phase2/sector4'
    ):
        raise ValueError("sector4.export_dir must be a canonical, safe relative Git directory.")

    repo_url = github.get("repo_url")
    if not isinstance(repo_url, str) or not repo_url.strip() or repo_url != repo_url.strip():
        raise ValueError("github.repo_url must be configured.")
    parsed = urlsplit(repo_url)
    github_remote = (
        parsed.scheme == "https"
        and parsed.netloc == "github.com"
        and parsed.username is None
        and parsed.password is None
        and not parsed.query
        and not parsed.fragment
        and re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", parsed.path)
        is not None
    )
    if not github_remote:
        if not allow_local:
            raise ValueError("Only credential-free HTTPS github.com repository URLs are allowed.")
        if parsed.scheme == "file" and not parsed.netloc and not parsed.query and not parsed.fragment:
            local_remote = Path(unquote(parsed.path)).resolve()
        elif not parsed.scheme and not parsed.netloc:
            local_remote = Path(repo_url).resolve()
        else:
            raise ValueError("The explicitly enabled test remote must be a local filesystem repository.")
        if not local_remote.is_dir():
            raise ValueError("The local test repository does not exist.")
        repo_url = str(local_remote)
    branch = github.get("branch", "main")
    if not isinstance(branch, str) or not branch or branch.startswith("-") or any(c in branch for c in "\r\n\x00"):
        raise ValueError("Invalid Git branch name.")

    artifacts = state.get("artifacts")
    if not isinstance(artifacts, list):
        raise ValueError("Sector 4 state must contain an artifact catalog.")
    relative_paths = []
    for artifact in artifacts:
        if not isinstance(artifact, dict):
            raise ValueError("Invalid Sector 4 artifact record.")
        relative = artifact.get("relative_path")
        source = _p3_safe_path(root, relative)
        if not source.is_file() or source.is_symlink():
            raise RuntimeError(f"Missing or unsafe Sector 4 artifact: {relative}")
        if _p3_file_hash(source) != artifact.get("sha256") or source.stat().st_size != artifact.get("bytes"):
            raise RuntimeError(f"Sector 4 artifact verification failed: {relative}")
        relative_paths.append(relative)
    if len(relative_paths) != len(set(relative_paths)):
        raise ValueError("Sector 4 artifact catalog has duplicate paths.")
    state_relative = "configs/sector4_state.json"
    state_source = _p3_safe_path(root, state_relative)
    if state_source.is_symlink() or json.loads(state_source.read_text(encoding="utf-8")) != state:
        raise RuntimeError("Sector 4 state changed during publisher verification.")
    if state_relative not in relative_paths:
        relative_paths.append(state_relative)
    obsolete_paths = state.get("obsolete_paths", [])
    if not isinstance(obsolete_paths, list) or any(not isinstance(p, str) for p in obsolete_paths):
        raise ValueError("Sector 4 obsolete_paths must be a list of recorded relative paths.")
    for relative in obsolete_paths:
        _p3_safe_path(root, relative)
        if not relative.startswith("results/") or relative in relative_paths:
            raise ValueError("Only obsolete recorded result files may be deleted from the export.")

    author_name = github.get("git_username", github.get("git_user_name", github.get("git_author_name", github.get("author_name", github.get("github_username")))))
    author_email = github.get("git_email", github.get("git_user_email", github.get("git_author_email", github.get("author_email"))))
    if not dry_run and push:
        if not isinstance(author_name, str) or not author_name.strip() or not isinstance(author_email, str) or not author_email.strip():
            raise ValueError("Configure github.git_username and github.git_email before publishing.")
        if any(c in author_name + author_email for c in "\r\n\x00"):
            raise ValueError("Git author name and email must be single-line values.")

    token = None
    secret_name = github.get("token_secret_name")
    if github_remote and secret_name:
        if not isinstance(secret_name, str):
            raise TypeError("github.token_secret_name must be a string.")
        token = os.environ.get(secret_name)
        if not token:
            try:
                from google.colab import userdata
                token = userdata.get(secret_name)
            except ImportError:
                pass
            except Exception as error:
                raise RuntimeError("The configured Colab GitHub secret could not be read.") from None
        if not token:
            raise RuntimeError("The configured GitHub token secret is unavailable.")

    outcome = {
        "status": "prepared",
        "branch": branch,
        "export_dir": export_dir,
        "count": len(relative_paths),
        "pushed": False,
        "dry_run": dry_run,
    }
    with tempfile.TemporaryDirectory(prefix="sector4-publish-") as temporary:
        temporary_root = Path(temporary)
        checkout = temporary_root / "repository"
        askpass = temporary_root / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "sys.stdout.write('x-access-token' if 'username' in prompt else os.environ.get('_P4_GITHUB_TOKEN', ''))\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        empty_hooks = temporary_root / "hooks"
        empty_hooks.mkdir()
        child_env = {key: value for key, value in os.environ.items()
                     if not key.startswith(('GIT_TRACE', 'GIT_CONFIG_KEY_', 'GIT_CONFIG_VALUE_'))
                     and key not in {'GIT_CURL_VERBOSE', 'SSH_ASKPASS'}}
        child_env.update({
            "GIT_TERMINAL_PROMPT": "0",
            "GIT_ASKPASS": str(askpass),
            "GIT_CONFIG_NOSYSTEM": "1",
            "GIT_CONFIG_GLOBAL": os.devnull,
            "GIT_LITERAL_PATHSPECS": "1",
        })
        for variable in ("GIT_DIR", "GIT_WORK_TREE", "GIT_INDEX_FILE", "GIT_CONFIG", "GIT_CONFIG_COUNT", "GIT_COMMON_DIR"):
            child_env.pop(variable, None)
        child_env.pop("_P4_GITHUB_TOKEN", None)
        if token:
            child_env["_P4_GITHUB_TOKEN"] = token

        def git(*args, cwd=None, allowed=(0,)):
            command = [
                "git", "-c", "credential.helper=", "-c", f"core.askPass={askpass}",
                "-c", f"core.hooksPath={empty_hooks}", *args,
            ]
            result = subprocess.run(
                command, cwd=cwd, env=child_env, text=True,
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False,
            )
            if result.returncode not in allowed:
                # Git output can include remote-supplied text: do not expose it or secrets.
                raise RuntimeError(f"Sector 4 Git {args[0]} failed (exit {result.returncode}).")
            return result

        git("check-ref-format", "--branch", branch)
        git("clone", "--quiet", "--single-branch", "--branch", branch, "--", repo_url, str(checkout))
        initial_tip = git("rev-parse", "HEAD", cwd=checkout).stdout.strip()

        def export_target(relative):
            relative_path = PurePosixPath(relative)
            if relative_path.is_absolute() or any(p in {".", "..", ".git"} for p in relative_path.parts) or "\\" in relative:
                raise ValueError("Invalid export artifact path.")
            target = checkout.joinpath(*export_parts.parts, *relative_path.parts)
            current = checkout
            for part in (*export_parts.parts, *relative_path.parts):
                current = current / part
                if current.is_symlink():
                    raise RuntimeError("The Git export path contains a symlink.")
            return target

        staged_paths = []
        for relative in relative_paths:
            source = _p3_safe_path(root, relative)
            target = export_target(relative)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
            record = next((item for item in artifacts if item["relative_path"] == relative), None)
            if record is not None and (
                _p3_file_hash(target) != record["sha256"]
                or target.stat().st_size != record["bytes"]
            ):
                raise RuntimeError("An exported artifact changed while it was being copied.")
            if relative == state_relative and json.loads(target.read_text(encoding="utf-8")) != state:
                raise RuntimeError("The exported state changed while it was being copied.")
            staged_paths.append(target.relative_to(checkout).as_posix())
        for relative in obsolete_paths:
            target = export_target(relative)
            git_relative = target.relative_to(checkout).as_posix()
            tracked = bool(git("ls-files", "--", git_relative, cwd=checkout).stdout.strip())
            if target.exists():
                if not target.is_file():
                    raise RuntimeError("An obsolete result path is not a file.")
                target.unlink()
            if tracked:
                staged_paths.append(git_relative)
        git("add", "--", *sorted(set(staged_paths)), cwd=checkout)
        changed = git("diff", "--cached", "--name-only", "-z", cwd=checkout).stdout
        changed_paths = [p for p in changed.split("\x00") if p]
        outcome["changed_paths"] = changed_paths
        if not changed_paths:
            outcome["status"] = "unchanged"
            return outcome
        # Re-read all catalog checks after copying to detect a concurrent local update.
        if _p4_read_state(root) != state:
            raise RuntimeError("Sector 4 changed during export; persist it again before publishing.")
        if dry_run:
            outcome["status"] = "dry_run"
            return outcome
        if not push:
            outcome["status"] = "prepared_no_push"
            return outcome
        git("-c", f"user.name={author_name}", "-c", f"user.email={author_email}",
            "commit", "--quiet", "-m", "Update Phase 2 Sector 4 split and domain-adaptation artifacts", cwd=checkout)
        commit = git("rev-parse", "HEAD", cwd=checkout).stdout.strip()
        git("fetch", "--quiet", "origin", f"refs/heads/{branch}", cwd=checkout)
        current_remote_tip = git("rev-parse", "FETCH_HEAD", cwd=checkout).stdout.strip()
        if current_remote_tip != initial_tip:
            raise RuntimeError("The remote branch changed during export; rerun publishing from its latest tip.")
        git("merge-base", "--is-ancestor", current_remote_tip, "HEAD", cwd=checkout)
        git("push", "--quiet", "origin", f"HEAD:refs/heads/{branch}", cwd=checkout)
        outcome.update({"status": "published", "pushed": True, "commit": commit})
        return outcome


### 4. Mount storage and load the completed Sector 3 tables

In [ ]:
if str(BOOTSTRAP["storage_root"]).startswith("/content/drive/"):
    from google.colab import drive
    drive.mount("/content/drive")

CONFIG, INPUTS, LINEAGE = io_load_sector4_context(BOOTSTRAP, SECTOR4_SETTINGS)

print("Sector 3 inputs verified:")
display(pd.DataFrame({"artifact": list(INPUTS), "rows": [len(table) for table in INPUTS.values()]}))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sector 3 inputs verified:


,artifact,rows
0,labelled_frame_manifest,47239
1,video_frame_manifest,117
2,video_class_matrix,43
3,video_class_support,117


### 5. Pure validation and reporting helpers


In [ ]:
# ------------------------------------------------------------------------------
# Phase 2 Sector 4: deterministic video splits and adaptation references.
#
# All functions are pure table transformations. No videos are decoded, no images
# are extracted, and no files are written. Video-level separation is audited;
# patient-level separation requires a separate trustworthy patient identifier.
# ------------------------------------------------------------------------------

S4_SPLIT_NAMES = ("train", "validation", "test")


S4_INPUT_NAMES = (
    "labelled_frame_manifest", "video_frame_manifest",
    "video_class_matrix", "video_class_support",
)


DOMAIN_ADAPTATION_DTYPES = {
    "video_key": "string",
    "video_relpath": "string",
    "opencv_frame_index": "Int64",
    "frame_number": "Int64",
    "frame_index_offset": "Int64",
    "source_timestamp_seconds": "Float64",
    "timestamp_basis": "string",
    "source_type": "string",
    "annotation_status": "string",
    "ground_truth_label": "string",
    "supervised_split": "string",
    "data_role": "string",
    "purpose": "string",
}


DOMAIN_ADAPTATION_COLUMNS = list(DOMAIN_ADAPTATION_DTYPES)


def s4_require_columns(table, columns, name):
    if not isinstance(table, pd.DataFrame):
        raise TypeError(f"{name} must be a pandas DataFrame.")
    if not table.columns.is_unique:
        raise ValueError(f"{name} contains duplicate column names.")
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")


def s4_valid_keys(values, name, unique=False):
    """Reject rather than coerce ambiguous identifiers or hidden whitespace."""
    if values.isna().any() or not values.map(
        lambda value: isinstance(value, str) and bool(value) and value == value.strip()
    ).all():
        raise ValueError(f"{name} requires non-empty canonical string identifiers.")
    if unique and not values.is_unique:
        raise ValueError(f"{name} contains duplicate identifiers.")
    return values.astype("string")


def s4_integer_series(values, name, minimum=0):
    if values.map(lambda value: isinstance(value, (bool, np.bool_))).any():
        raise ValueError(f"{name} cannot contain boolean values.")
    numbers = pd.to_numeric(values, errors="coerce")
    floating = numbers.astype("float64")
    valid = numbers.notna() & np.isfinite(floating) & numbers.mod(1).eq(0)
    valid &= floating.abs().le(2**53 - 1)
    if minimum is not None:
        valid &= numbers.ge(minimum)
    if not valid.all():
        raise ValueError(f"{name} requires finite integral values" +
                         (f" >= {minimum}." if minimum is not None else "."))
    return numbers.astype("Int64")


def s4_boolean(value, name):
    if not isinstance(value, (bool, np.bool_)):
        raise ValueError(f"{name} must be a genuine boolean, not {value!r}.")
    return bool(value)


def s4_boolean_series(values, name):
    if values.isna().any() or not values.map(
        lambda value: isinstance(value, (bool, np.bool_))
    ).all():
        raise ValueError(f"{name} requires genuine non-null boolean values.")
    return values.astype(bool)


def validate_domain_adaptation_configuration(config):
    switches = (
        "domain_adaptation_enabled",
        "domain_adaptation_include_fully_unlabelled_videos",
        "domain_adaptation_include_train_video_unlabelled_frames",
    )
    required = {*switches, "domain_adaptation_sampling_seconds"}
    missing = sorted(required - set(config))
    if missing:
        raise KeyError(f"Domain-adaptation configuration is missing keys: {missing}")
    result = {name: s4_boolean(config[name], name) for name in switches}
    seconds = config["domain_adaptation_sampling_seconds"]
    if isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, Real):
        raise ValueError("domain_adaptation_sampling_seconds must be a finite positive number.")
    seconds = float(seconds)
    if not np.isfinite(seconds) or seconds <= 0:
        raise ValueError("domain_adaptation_sampling_seconds must be a finite positive number.")
    result["domain_adaptation_sampling_seconds"] = seconds
    return result


def s4_validate_inputs(inputs):
    """Validate Sector 3 evidence and return canonical copies plus binary matrix."""
    missing = sorted(set(S4_INPUT_NAMES) - set(inputs))
    if missing:
        raise KeyError(f"Sector 4 inputs are missing: {missing}")
    for name in S4_INPUT_NAMES:
        if not isinstance(inputs[name], pd.DataFrame):
            raise TypeError(f"{name} must be a pandas DataFrame.")
    labelled = inputs["labelled_frame_manifest"].copy()
    videos = inputs["video_frame_manifest"].copy()
    matrix = inputs["video_class_matrix"].copy()
    support = inputs["video_class_support"].copy()
    s4_require_columns(labelled, ["video_key", "opencv_frame_index", "frame_number",
                                 "frame_index_offset", "finding_class_normalized"], "labelled_frame_manifest")
    s4_require_columns(videos, ["video_key", "video_relpath", "video_annotation_type",
                               "frame_count", "frame_count_source", "container_fps"], "video_frame_manifest")
    s4_require_columns(matrix, ["video_key"], "video_class_matrix")
    s4_require_columns(support, ["video_key", "original_annotation_count",
                                "retained_annotation_count"], "video_class_support")
    if labelled.empty or videos.empty:
        raise ValueError("Sector 4 requires a non-empty accepted manifest and physical video inventory.")
    for table, name, unique in ((labelled, "labelled_frame_manifest", False),
                                (videos, "video_frame_manifest", True),
                                (matrix, "video_class_matrix", True),
                                (support, "video_class_support", True)):
        table["video_key"] = s4_valid_keys(table["video_key"], name + ".video_key", unique)
    physical_keys = set(videos["video_key"])
    if set(support["video_key"]) != physical_keys:
        raise ValueError("video_class_support must cover exactly the physical video inventory.")
    if not set(labelled["video_key"]).issubset(physical_keys):
        raise ValueError("Accepted annotations reference videos outside the physical inventory.")
    classes = sorted(set(matrix.columns) - {"video_key"})
    if not classes or not all(isinstance(name, str) and name.strip() == name and name for name in classes):
        raise ValueError("video_class_matrix requires non-empty canonical class column names.")
    if matrix[classes].isna().any().any() or not matrix[classes].isin([0, 1]).all().all():
        raise ValueError("video_class_matrix must contain only non-null binary label values 0 or 1.")
    labelled["finding_class_normalized"] = s4_valid_keys(
        labelled["finding_class_normalized"], "labelled_frame_manifest.finding_class_normalized")
    if not set(labelled["finding_class_normalized"]).issubset(classes):
        raise ValueError("The video-class matrix omits classes present in accepted annotations.")
    for column in ("opencv_frame_index", "frame_number", "frame_index_offset"):
        labelled[column] = s4_integer_series(labelled[column], "labelled_frame_manifest." + column,
                                            minimum=None if column == "frame_index_offset" else 0)
    if not labelled["opencv_frame_index"].eq(labelled["frame_number"] + labelled["frame_index_offset"]).all():
        raise ValueError("Accepted frame indices disagree with their source numbers and per-video offsets.")
    if labelled.groupby("video_key", observed=True)["frame_index_offset"].nunique().gt(1).any():
        raise ValueError("Accepted annotations have inconsistent offsets within a video.")
    if "annotation_id" in labelled:
        labelled["annotation_id"] = s4_valid_keys(labelled["annotation_id"], "annotation_id", True)
    if "annotation_status" in labelled and not labelled["annotation_status"].eq(
        "source_expert_annotation_alignment_confirmed"
    ).all():
        raise ValueError("The accepted manifest contains annotations without confirmed expert alignment.")
    if "alignment_training_eligible" in labelled and not s4_boolean_series(
        labelled["alignment_training_eligible"], "alignment_training_eligible"
    ).all():
        raise ValueError("The accepted manifest contains ineligible alignment evidence.")
    videos["video_relpath"] = s4_valid_keys(videos["video_relpath"], "video_relpath")
    def portable_path(value):
        path = PurePosixPath(value)
        return not path.is_absolute() and "\\" not in value and not any(
            part in {".", "..", ""} for part in value.split("/")
        ) and path.as_posix() == value
    if not videos["video_relpath"].map(portable_path).all():
        raise ValueError("video_relpath must be a canonical storage-root-relative POSIX path.")
    if "video_relpath" in labelled:
        expected_paths = labelled["video_key"].map(videos.set_index("video_key")["video_relpath"])
        if not labelled["video_relpath"].eq(expected_paths).all():
            raise ValueError("Accepted annotation paths disagree with the physical video manifest.")
    videos["frame_count"] = s4_integer_series(videos["frame_count"], "decoded frame_count", minimum=1)
    if not videos["frame_count_source"].eq("completed_sector2_decode_audit").all():
        raise ValueError("frame_count must come from the completed Sector 2 decode audit, not a container estimate.")
    if "frames_read_before_stop" in videos:
        counts = s4_integer_series(videos["frames_read_before_stop"], "frames_read_before_stop", minimum=1)
        if not counts.eq(videos["frame_count"]).all():
            raise ValueError("Decoded frame counts disagree with frames_read_before_stop.")
    if "decoding_completed" in videos and not s4_boolean_series(videos["decoding_completed"], "decoding_completed").all():
        raise ValueError("Every physical video requires a completed decode audit.")
    if "stop_reason" in videos and not videos["stop_reason"].eq("read_completed").all():
        raise ValueError("Every video must have stop_reason='read_completed'.")
    if videos["container_fps"].map(lambda value: isinstance(value, (bool, np.bool_))).any():
        raise ValueError("container_fps cannot contain boolean values.")
    fps = pd.to_numeric(videos["container_fps"], errors="coerce")
    if fps.isna().any() or not (np.isfinite(fps.astype("float64")) & fps.gt(0)).all():
        raise ValueError("container_fps must be a finite positive nominal frame rate.")
    videos["container_fps"] = fps.astype("Float64")
    bounds = labelled["video_key"].map(videos.set_index("video_key")["frame_count"])
    if not labelled["opencv_frame_index"].lt(bounds).all():
        raise ValueError("Accepted frame indices exceed decoded video bounds.")
    for column in ("original_annotation_count", "retained_annotation_count"):
        support[column] = s4_integer_series(support[column], "video_class_support." + column)
    if support["retained_annotation_count"].gt(support["original_annotation_count"]).any():
        raise ValueError("Retained annotation counts cannot exceed original annotation counts.")
    actual_counts = labelled.groupby("video_key", observed=True).size()
    expected_counts = support["video_key"].map(actual_counts).fillna(0).astype("Int64")
    if not support["retained_annotation_count"].eq(expected_counts).all():
        raise ValueError("video_class_support.retained_annotation_count disagrees with accepted manifest rows.")
    count_map = support.set_index("video_key")["original_annotation_count"]
    original_counts = videos["video_key"].map(count_map)
    expected_types = np.where(original_counts.gt(0), "partially_labelled", "fully_unlabelled")
    if not videos["video_annotation_type"].eq(expected_types).all():
        raise ValueError("Video annotation types disagree with original annotation counts; excluded evidence cannot be reclassified as fully unlabelled.")
    originally_labelled_keys = set(support.loc[support["original_annotation_count"].gt(0), "video_key"])
    if set(matrix["video_key"]) != originally_labelled_keys:
        raise ValueError("Sector 3 video_class_matrix must cover exactly the originally labelled videos, including zero retained-support rows.")
    indexed = matrix.set_index("video_key")[classes].sort_index().astype("int8")
    pairs = labelled[["video_key", "finding_class_normalized"]].drop_duplicates()
    actual_binary = pd.crosstab(pairs["video_key"], pairs["finding_class_normalized"]).gt(0).astype("int8")
    actual_binary = actual_binary.reindex(index=indexed.index, columns=classes, fill_value=0)
    if not indexed.equals(actual_binary.rename_axis(columns=None)):
        # DataFrame.equals includes axis names; compare the validated axes and data directly.
        if not np.array_equal(indexed.to_numpy(), actual_binary.to_numpy()):
            raise ValueError("video_class_matrix disagrees with accepted per-video clinical class evidence.")
    accepted_keys = set(labelled["video_key"])
    indexed = indexed.loc[indexed.sum(axis=1).gt(0)]
    if set(indexed.index) != accepted_keys:
        raise ValueError("Retained video-class matrix keys must exactly equal accepted manifest video keys.")
    unique_frames = labelled[["video_key", "opencv_frame_index"]].drop_duplicates().groupby("video_key", observed=True).size()
    class_counts = labelled.groupby("video_key", observed=True)["finding_class_normalized"].nunique()
    for column, values in (("retained_unique_frame_count", unique_frames), ("retained_class_count", class_counts)):
        expected = support["video_key"].map(values).fillna(0).astype("Int64")
        if column in support and not s4_integer_series(support[column], column).eq(expected).all():
            raise ValueError(f"video_class_support.{column} disagrees with accepted evidence.")
        support[column] = expected
    for column, expected in (("source_has_annotations", support["original_annotation_count"].gt(0)),
                             ("has_accepted_labelled_evidence", support["retained_annotation_count"].gt(0))):
        if column in support and not s4_boolean_series(support[column], column).eq(expected).all():
            raise ValueError(f"video_class_support.{column} disagrees with annotation counts.")
        support[column] = expected.astype(bool)
    # annotation_id is stable in Sector 3; a row hash supplies deterministic ties
    # for simplified compatible fixtures that omit that optional identity.
    sort_columns = ["video_key", "opencv_frame_index", "finding_class_normalized"]
    if "annotation_id" in labelled:
        sort_columns.append("annotation_id")
    else:
        labelled["_s4_sort_hash"] = pd.util.hash_pandas_object(
            labelled.reindex(sorted(labelled.columns), axis=1).astype("string"), index=False)
        sort_columns.append("_s4_sort_hash")
    labelled = labelled.sort_values(sort_columns, kind="stable").drop(columns="_s4_sort_hash", errors="ignore").reset_index(drop=True)
    return labelled, videos.sort_values("video_key", kind="stable").reset_index(drop=True), indexed, support.sort_values("video_key", kind="stable").reset_index(drop=True)


def s4_validate_domain_manifest(domain_manifest, source_videos, labelled, videos_with_roles):
    s4_require_columns(domain_manifest, DOMAIN_ADAPTATION_COLUMNS, "domain_adaptation_manifest")
    if not domain_manifest["ground_truth_label"].isna().all():
        raise RuntimeError("Unlabelled domain references cannot contain ground-truth or pseudo-labels.")
    if not domain_manifest["annotation_status"].eq("unlabelled").all():
        raise RuntimeError("Domain adaptation references must preserve unlabelled annotation status.")
    if domain_manifest.empty:
        return
    if not set(domain_manifest["video_key"]).issubset(set(source_videos["video_key"])):
        raise RuntimeError("Domain references include a video outside the eligible source pool.")
    roles = videos_with_roles.set_index("video_key")
    forbidden_keys = set(roles.index[roles["supervised_split"].isin(["validation", "test"])])
    if set(domain_manifest["video_key"]) & forbidden_keys:
        raise RuntimeError("Validation/test videos leaked into domain adaptation.")
    pairs = domain_manifest[["video_key", "opencv_frame_index"]]
    overlap = pairs.merge(labelled[["video_key", "opencv_frame_index"]].drop_duplicates(),
                          on=["video_key", "opencv_frame_index"], how="inner")
    if not overlap.empty:
        raise RuntimeError("Accepted labelled frames leaked into the unlabelled adaptation pool.")
    bounds = domain_manifest["video_key"].map(roles["frame_count"])
    if not (domain_manifest["opencv_frame_index"].ge(0) & domain_manifest["opencv_frame_index"].lt(bounds)).all():
        raise RuntimeError("Domain-adaptation references exceed completed decoded frame bounds.")
    train = domain_manifest["source_type"].eq("unlabelled_frame_from_train_video")
    fully = domain_manifest["source_type"].eq("fully_unlabelled_video")
    if not (train | fully).all() or not domain_manifest.loc[train, "supervised_split"].eq("train").all():
        raise RuntimeError("Domain-adaptation source types contradict their supervised split.")
    if not domain_manifest.loc[fully, ["frame_number", "frame_index_offset", "supervised_split"]].isna().all().all():
        raise RuntimeError("Fully unlabelled sources must not invent verified metadata numbering or supervised splits.")
    train_table = domain_manifest.loc[train]
    if not train_table["frame_number"].ge(0).all() or not train_table["opencv_frame_index"].eq(
        train_table["frame_number"] + train_table["frame_index_offset"]
    ).all():
        raise RuntimeError("Train source references disagree with their verified frame numbering.")
    train_keys = set(train_table["video_key"])
    if any(int(roles.loc[key, "original_annotation_count"]) != int(roles.loc[key, "retained_annotation_count"]) for key in train_keys):
        raise RuntimeError("Train videos with unverified source annotations leaked into adaptation.")


def s4_build_reports(labelled, matrix, split_manifest, videos, domain_sources, domain_manifest, config):
    split_settings = validate_split_configuration(config)
    role_report = videos.groupby(["video_annotation_type", "supervised_split", "data_role"],
                                dropna=False, observed=True).size().rename("video_count").reset_index()
    role_report = role_report.sort_values(["video_annotation_type", "supervised_split", "data_role"], kind="stable", na_position="last").reset_index(drop=True)
    summary_rows = []
    class_rows = []
    unique_frames = labelled[["video_key", "opencv_frame_index", "split"]].drop_duplicates()
    class_pairs = labelled[["video_key", "opencv_frame_index", "finding_class_normalized", "split"]].drop_duplicates()
    for split in S4_SPLIT_NAMES:
        keys = split_manifest.loc[split_manifest["split"].eq(split), "video_key"]
        annotation_count = int(labelled["split"].eq(split).sum())
        frame_count = int(unique_frames["split"].eq(split).sum())
        summary_rows.append({"split": split, "requested_video_fraction": split_settings[split],
                             "video_count": len(keys), "actual_video_fraction": len(keys) / len(matrix),
                             "retained_annotation_count": annotation_count,
                             "retained_unique_frame_count": frame_count,
                             "decoded_video_frame_count": int(videos.loc[videos["video_key"].isin(keys), "frame_count"].sum()),
                             "partition_validation_passed": True})
    split_summary = pd.DataFrame(summary_rows)
    for clinical_class in matrix.columns:
        total_support = int(matrix[clinical_class].sum())
        per_split_support = {split: int(matrix.loc[split_manifest.loc[split_manifest["split"].eq(split), "video_key"], clinical_class].sum())
                             for split in S4_SPLIT_NAMES}
        all_covered = all(count > 0 for count in per_split_support.values())
        coverage = ("no_retained_support" if total_support == 0 else
                    "fewer_than_three_supporting_videos" if total_support < 3 else
                    "represented_in_all_splits" if all_covered else "missing_from_at_least_one_split")
        for split in S4_SPLIT_NAMES:
            clinical_mask = labelled["finding_class_normalized"].eq(clinical_class) & labelled["split"].eq(split)
            unique_mask = class_pairs["finding_class_normalized"].eq(clinical_class) & class_pairs["split"].eq(split)
            class_rows.append({"finding_class_normalized": clinical_class, "split": split,
                               "supporting_video_count": per_split_support[split],
                               "retained_annotation_count": int(clinical_mask.sum()),
                               "retained_unique_frame_count": int(unique_mask.sum()),
                               "total_supporting_video_count": total_support,
                               "absent_in_split": per_split_support[split] == 0,
                               "scarce_video_support": total_support < 3,
                               "represented_in_all_splits": all_covered,
                               "coverage_status": coverage,
                               "stratification_guarantees_class_coverage": False})
    class_report = pd.DataFrame(class_rows)
    domain_rows = []
    for source_type, annotation_type in (("fully_unlabelled_video", "fully_unlabelled"),
                                        ("unlabelled_frame_from_train_video", "partially_labelled")):
        source_keys = domain_sources.loc[domain_sources["video_annotation_type"].eq(annotation_type), "video_key"]
        references = domain_manifest.loc[domain_manifest["source_type"].eq(source_type)]
        domain_rows.append({"source_type": source_type, "eligible_video_count": len(source_keys),
                            "sampled_video_count": references["video_key"].nunique(),
                            "frame_reference_count": len(references), "unlabelled_only_validation_passed": True})
    eligibility_columns = ["video_key", "video_annotation_type", "supervised_split", "data_role",
                           "original_annotation_count", "retained_annotation_count", "excluded_annotation_count",
                           "domain_sampling_requested", "domain_sampling_eligible", "domain_sampling_reason"]
    return {"video_role_report": role_report, "split_summary_report": split_summary,
            "class_split_support_report": class_report,
            "domain_adaptation_summary": pd.DataFrame(domain_rows),
            "domain_eligibility_report": videos[eligibility_columns].copy()}


### 6. Multilabel-stratified video split

Normalize the saved `video_key` column into a sorted index, validate binary class presence against accepted annotations, and split only videos with retained evidence. Two-stage stratification targets the configured fractions; actual counts are reported.

In [ ]:
def validate_split_configuration(config):
    required = {"train_fraction", "validation_fraction", "test_fraction", "split_strategy", "seed"}
    missing = sorted(required - set(config))
    if missing:
        raise KeyError(f"Split configuration is missing keys: {missing}")
    fractions = {}
    for split in S4_SPLIT_NAMES:
        value = config[f"{split}_fraction"]
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, Real):
            raise ValueError(f"{split}_fraction must be a finite numeric fraction.")
        value = float(value)
        if not np.isfinite(value) or not 0 < value < 1:
            raise ValueError(f"{split}_fraction must be finite and strictly between 0 and 1.")
        fractions[split] = value
    if not np.isclose(sum(fractions.values()), 1.0, rtol=0.0, atol=1e-12):
        raise ValueError("train_fraction + validation_fraction + test_fraction must equal 1.0.")
    if config["split_strategy"] != "video_level_multilabel_stratified":
        raise ValueError(f"Unsupported split strategy: {config['split_strategy']!r}")
    seed = config["seed"]
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, Integral):
        raise ValueError("The split seed must be an integer; floats and booleans are not accepted.")
    if not 0 <= int(seed) <= 2**32 - 1:
        raise ValueError("The split seed must be in the interval [0, 2**32 - 1].")
    return {**fractions, "seed": int(seed)}


def split_labelled_videos(video_class_matrix, config):
    """Two-stage multilabel stratification on canonically ordered retained videos.

    This is a heuristic distribution objective, not a guarantee that every
    class appears in all splits, especially when fewer than three videos carry it.
    """
    settings = validate_split_configuration(config)
    matrix = video_class_matrix.copy()
    if "video_key" in matrix:
        s4_valid_keys(matrix["video_key"], "video_class_matrix.video_key", True)
        matrix = matrix.set_index("video_key")
    else:
        s4_valid_keys(pd.Series(matrix.index), "video_class_matrix.index", True)
    if len(matrix) < 3 or matrix.shape[1] == 0:
        raise ValueError("At least three retained-evidence videos and one class are required for three non-empty splits.")
    if matrix.isna().any().any() or not matrix.isin([0, 1]).all().all():
        raise ValueError("The retained video-class matrix requires binary non-null values.")
    if matrix.sum(axis=1).eq(0).any():
        raise ValueError("Only videos with retained labelled evidence can receive supervised splits.")
    matrix = matrix.sort_index().reindex(sorted(matrix.columns), axis=1)
    labels = matrix.to_numpy(dtype=np.int8)
    features = np.zeros((len(matrix), 1), dtype=np.int8)
    temporary_fraction = settings["validation"] + settings["test"]
    try:
        first = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=temporary_fraction, random_state=settings["seed"])
        train, temporary = next(first.split(features, labels))
        if len(train) == 0 or len(temporary) < 2:
            raise ValueError("The first stage did not leave enough videos for validation and test.")
        second = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=settings["test"] / temporary_fraction, random_state=settings["seed"])
        validation_relative, test_relative = next(second.split(features[temporary], labels[temporary]))
    except ValueError as error:
        raise ValueError("Configured split fractions cannot produce three non-empty video splits for the retained evidence pool.") from error
    keys = matrix.index.to_numpy()
    result = {"train": set(keys[train]), "validation": set(keys[temporary[validation_relative]]), "test": set(keys[temporary[test_relative]])}
    validate_video_splits(result, matrix.index)
    return result


### 7. Propagate split

Require three nonempty, disjoint splits that cover the accepted-evidence video pool exactly. Build the lookup used to propagate each video's split to its annotations.

In [ ]:
def validate_video_splits(video_splits, expected_video_keys):
    if set(video_splits) != set(S4_SPLIT_NAMES):
        raise KeyError(f"Video splits require exactly these names: {S4_SPLIT_NAMES}")
    partitions = {name: set(video_splits[name]) for name in S4_SPLIT_NAMES}
    if any(not keys for keys in partitions.values()):
        raise RuntimeError("Every supervised split must contain at least one video.")
    for first, second in (("train", "validation"), ("train", "test"), ("validation", "test")):
        overlap = partitions[first] & partitions[second]
        if overlap:
            raise RuntimeError(f"Video leakage between {first} and {second}: {sorted(overlap)}")
    assigned = set.union(*partitions.values())
    expected = set(expected_video_keys)
    if assigned != expected:
        raise RuntimeError(f"Video partition coverage mismatch. Missing: {sorted(expected - assigned)}. Unexpected: {sorted(assigned - expected)}.")


def build_split_lookup(video_splits):
    return {key: name for name in S4_SPLIT_NAMES for key in sorted(video_splits[name])}


### 8. Assign roles to all videos

Cross-check original/retained annotation counts. Videos with no accepted labelled evidence remain `labelled_evidence_excluded`. Train videos with rejected source annotations keep their supervised role and are blocked from adaptation.

In [ ]:
def derive_video_role(annotation_type, supervised_split, config,
                      original_annotation_count=None, retained_annotation_count=None):
    """Return actual allowed role and sampling eligibility, with a guard reason."""
    settings = validate_domain_adaptation_configuration(config)
    if original_annotation_count is None or retained_annotation_count is None:
        raise ValueError("Original and retained annotation counts are required to determine a safe video role.")
    counts = s4_integer_series(pd.Series([original_annotation_count, retained_annotation_count]),
                               "original and retained annotation counts")
    original, retained = (int(value) for value in counts)
    if retained > original:
        raise ValueError("Retained annotation count cannot exceed the original annotation count.")
    has_split = pd.notna(supervised_split)
    enabled = settings["domain_adaptation_enabled"]
    if annotation_type == "fully_unlabelled":
        if original != 0 or retained != 0 or has_split:
            raise ValueError("A fully unlabelled source must have zero original and retained annotations and no supervised split.")
        requested = enabled and settings["domain_adaptation_include_fully_unlabelled_videos"]
        reason = "fully_unlabelled_source" if requested else (
            "domain_adaptation_disabled" if not enabled else "fully_unlabelled_source_disabled")
        return {"data_role": "domain_adaptation_only" if requested else "unlabelled_reserved",
                "domain_sampling_requested": requested, "domain_sampling_eligible": requested,
                "domain_sampling_reason": reason}
    if annotation_type != "partially_labelled" or original <= 0:
        raise ValueError("Unsupported or contradictory video annotation type.")
    if retained == 0:
        if has_split:
            raise ValueError("A video without retained labelled evidence cannot have a supervised split.")
        return {"data_role": "labelled_evidence_excluded", "domain_sampling_requested": False,
                "domain_sampling_eligible": False, "domain_sampling_reason": "labelled_evidence_excluded"}
    if not has_split or supervised_split not in S4_SPLIT_NAMES:
        raise ValueError("Every retained-evidence video requires a valid supervised split.")
    if supervised_split != "train":
        return {"data_role": "supervised_" + supervised_split, "domain_sampling_requested": False,
                "domain_sampling_eligible": False, "domain_sampling_reason": "supervised_validation_or_test"}
    requested = enabled and settings["domain_adaptation_include_train_video_unlabelled_frames"]
    eligible = requested and original == retained
    reason = ("unverified_source_annotations" if original > retained else
              "verified_train_source" if eligible else
              "domain_adaptation_disabled" if not enabled else "train_unlabelled_source_disabled")
    return {"data_role": "supervised_train_plus_domain_adaptation" if eligible else "supervised_train",
            "domain_sampling_requested": requested, "domain_sampling_eligible": eligible,
            "domain_sampling_reason": reason}


def assign_video_roles(video_manifest, split_lookup, config, video_class_support):
    support_columns = ["video_key", "original_annotation_count", "retained_annotation_count",
                       "retained_unique_frame_count", "retained_class_count"]
    # Re-running cells must replace their derived columns rather than suffix them.
    derived = [*support_columns[1:], "supervised_split", "data_role", "domain_sampling_requested",
               "domain_sampling_eligible", "domain_sampling_reason", "excluded_annotation_count", "frame_index_offset"]
    result = video_manifest.drop(columns=derived, errors="ignore").merge(
        video_class_support[support_columns], on="video_key", how="left", validate="one_to_one")
    result["supervised_split"] = result["video_key"].map(split_lookup).astype("string")
    roles = [derive_video_role(row.video_annotation_type, row.supervised_split, config,
                               row.original_annotation_count, row.retained_annotation_count)
             for row in result.itertuples(index=False)]
    role_table = pd.DataFrame(roles, index=result.index)
    for column in role_table:
        result[column] = role_table[column]
    result["excluded_annotation_count"] = result["original_annotation_count"] - result["retained_annotation_count"]
    return result.sort_values("video_key", kind="stable").reset_index(drop=True)


### 9. Identify domain-adaptation source videos

Use confirmed decoded frame identities to exclude accepted labelled frames, and select only roles that also pass the conservative eligibility guard.

In [ ]:
def build_verified_frame_lookup(labelled_manifest):
    s4_require_columns(labelled_manifest, ["video_key", "opencv_frame_index"], "labelled_manifest")
    indices = s4_integer_series(labelled_manifest["opencv_frame_index"], "accepted opencv_frame_index")
    return labelled_manifest.assign(opencv_frame_index=indices).groupby("video_key", observed=True)[
        "opencv_frame_index"].agg(lambda values: frozenset(int(value) for value in values)).to_dict()


def select_domain_adaptation_videos(video_manifest, config):
    settings = validate_domain_adaptation_configuration(config)
    s4_require_columns(video_manifest, ["video_key", "data_role", "domain_sampling_eligible"], "video_manifest_with_roles")
    eligible = s4_boolean_series(video_manifest["domain_sampling_eligible"], "domain_sampling_eligible")
    if not settings["domain_adaptation_enabled"]:
        return video_manifest.iloc[0:0].copy().reset_index(drop=True)
    allowed = video_manifest["data_role"].isin({"domain_adaptation_only", "supervised_train_plus_domain_adaptation"})
    if (eligible & ~allowed).any():
        raise ValueError("Sampling eligibility contradicts an allowed domain-adaptation role.")
    return video_manifest.loc[eligible & allowed].sort_values("video_key", kind="stable").reset_index(drop=True).copy()


### 10. Generate domain-adaptation references

Sample decoded indices using nominal FPS and the configured interval. Use per-video verified offsets for train sources. Keep metadata numbering null when no verified mapping exists.

In [ ]:
def get_domain_sampling_stride(config, video_fps):
    seconds = validate_domain_adaptation_configuration(config)["domain_adaptation_sampling_seconds"]
    if isinstance(video_fps, (bool, np.bool_)):
        raise ValueError("Video FPS must be a finite positive nominal frame rate.")
    video_fps = float(video_fps)
    if not np.isfinite(video_fps) or video_fps <= 0:
        raise ValueError("Video FPS must be a finite positive nominal frame rate.")
    frame_interval = seconds * video_fps
    if not np.isfinite(frame_interval) or frame_interval > np.iinfo(np.int64).max:
        raise ValueError("The domain-adaptation interval cannot be represented as a frame stride.")
    return max(1, int(round(frame_interval)))


def empty_domain_adaptation_manifest():
    return pd.DataFrame({column: pd.Series(dtype=dtype) for column, dtype in DOMAIN_ADAPTATION_DTYPES.items()})


def build_domain_records_for_video(video, verified_frame_lookup, config, frame_index_offset=None):
    """Build sampled references using decoded bounds and each video's own offset.

    Fully unlabelled sources have no verified metadata numbering: frame_number
    and frame_index_offset remain null. Timestamps are nominal FPS estimates.
    """
    required = {"video_key", "video_relpath", "video_annotation_type", "supervised_split",
                "data_role", "domain_sampling_eligible", "original_annotation_count",
                "retained_annotation_count", "frame_count", "container_fps"}
    missing = sorted(required - set(video.index))
    if missing:
        raise KeyError(f"Domain source video is missing fields: {missing}")
    derived = derive_video_role(video["video_annotation_type"], video["supervised_split"], config,
                                video["original_annotation_count"], video["retained_annotation_count"])
    if not s4_boolean(video["domain_sampling_eligible"], "domain_sampling_eligible") or not derived["domain_sampling_eligible"] or derived["data_role"] != video["data_role"]:
        raise ValueError(f"Video is not safely eligible for domain adaptation: {video['video_key']}")
    fully_unlabelled = video["video_annotation_type"] == "fully_unlabelled"
    frame_count = int(s4_integer_series(pd.Series([video["frame_count"]]), "source decoded frame_count", 1).iloc[0])
    fps = float(video["container_fps"])
    stride = get_domain_sampling_stride(config, fps)
    indices = np.arange(0, frame_count, stride, dtype=np.int64)
    verified = np.asarray(sorted(verified_frame_lookup.get(video["video_key"], frozenset())), dtype=np.int64)
    if fully_unlabelled and verified.size:
        raise ValueError("A fully unlabelled source cannot contain accepted labelled frame references.")
    if not fully_unlabelled and verified.size == 0:
        raise ValueError("A train video requires its accepted frame lookup before domain sampling.")
    if verified.size:
        indices = indices[~np.isin(indices, verified)]
    if fully_unlabelled:
        if frame_index_offset is not None and pd.notna(frame_index_offset):
            raise ValueError("A fully unlabelled video has no verified metadata frame offset.")
        offset = pd.NA
        frame_numbers = pd.array([pd.NA] * len(indices), dtype="Int64")
    else:
        if frame_index_offset is None:
            frame_index_offset = video.get("frame_index_offset", pd.NA)
        if pd.isna(frame_index_offset):
            raise ValueError("A train source requires its own verified per-video frame offset.")
        offset = int(s4_integer_series(pd.Series([frame_index_offset]), "per-video frame offset", None).iloc[0])
        frame_numbers = indices - offset
        valid = frame_numbers >= 0
        indices, frame_numbers = indices[valid], frame_numbers[valid]
    records = pd.DataFrame({
        "video_key": video["video_key"], "video_relpath": video["video_relpath"],
        "opencv_frame_index": indices, "frame_number": frame_numbers,
        "frame_index_offset": offset, "source_timestamp_seconds": indices / fps,
        "timestamp_basis": "nominal_container_fps_estimate",
        "source_type": "fully_unlabelled_video" if fully_unlabelled else "unlabelled_frame_from_train_video",
        "annotation_status": "unlabelled", "ground_truth_label": pd.NA,
        "supervised_split": pd.NA if fully_unlabelled else "train",
        "data_role": video["data_role"], "purpose": "visual_domain_adaptation",
    })
    return records.astype(DOMAIN_ADAPTATION_DTYPES)


### 11. Build domain-adaptation manifest

Build typed frame-reference tables, reject duplicates and forbidden sources, and validate that no accepted expert frame or evaluation video entered the adaptation pool. The orchestration function also builds all reports.

In [ ]:
def build_domain_adaptation_manifest(domain_source_videos, verified_frame_lookup, config):
    if domain_source_videos.empty:
        return empty_domain_adaptation_manifest()
    s4_valid_keys(domain_source_videos["video_key"], "domain_source_videos.video_key", True)
    tables = [build_domain_records_for_video(video, verified_frame_lookup, config)
              for _, video in domain_source_videos.sort_values("video_key", kind="stable").iterrows()]
    tables = [table for table in tables if not table.empty]
    if not tables:
        return empty_domain_adaptation_manifest()
    # All columns already have one explicit nullable schema, including all-null
    # metadata fields; concatenation therefore does not depend on null inference.
    result = pd.concat(tables, ignore_index=True).astype(DOMAIN_ADAPTATION_DTYPES)
    if result.duplicated(["video_key", "opencv_frame_index"]).any():
        raise RuntimeError("Duplicate domain-adaptation frame references found.")
    return result.sort_values(["video_key", "opencv_frame_index"], kind="stable").reset_index(drop=True)


def build_sector4_tables(inputs, config):
    """Build and audit every Sector 4 result from verified Sector 3 tables."""
    validate_split_configuration(config)
    validate_domain_adaptation_configuration(config)
    labelled, videos, matrix, support = s4_validate_inputs(inputs)
    video_splits = split_labelled_videos(matrix, config)
    lookup = build_split_lookup(video_splits)
    labelled = labelled.assign(split=labelled["video_key"].map(lookup).astype("string"))
    if labelled["split"].isna().any():
        raise RuntimeError("Some accepted annotations did not receive a supervised split.")
    videos = assign_video_roles(videos, lookup, config, support)
    offsets = labelled.groupby("video_key", observed=True)["frame_index_offset"].first()
    videos["frame_index_offset"] = videos["video_key"].map(offsets).astype("Int64")
    split_manifest = videos.loc[videos["supervised_split"].notna(), [
        "video_key", "video_relpath", "supervised_split", "frame_index_offset",
        "retained_annotation_count", "retained_unique_frame_count", "retained_class_count",
    ]].rename(columns={"supervised_split": "split"}).reset_index(drop=True)
    verified_lookup = build_verified_frame_lookup(labelled)
    domain_sources = select_domain_adaptation_videos(videos, config)
    domain_manifest = build_domain_adaptation_manifest(domain_sources, verified_lookup, config)
    s4_validate_domain_manifest(domain_manifest, domain_sources, labelled, videos)
    validate_video_splits(video_splits, labelled["video_key"].unique())
    if len(labelled) != int(support["retained_annotation_count"].sum()) or len(videos) != len(support):
        raise RuntimeError("Sector 4 annotation or physical-video coverage changed during transformation.")
    tables = {"video_split_manifest": split_manifest,
              "labelled_frame_manifest_with_split": labelled,
              "video_manifest_with_roles": videos,
              "domain_source_videos": domain_sources,
              "domain_adaptation_manifest": domain_manifest}
    tables.update(s4_build_reports(labelled, matrix, split_manifest, videos, domain_sources, domain_manifest, config))
    return tables


### 12. Reuse verified saved transformations or compute changed inputs

Active core functions/constants and dependency versions participate in the cache
identity. Changing only a clock value does not rerun transformations. Configuration
inheritance reports always reflect this invocation's effective authored values.

In [ ]:


def _s4_core_code_value(code):
    """Describe executable code without filenames, source lines or debug positions."""
    fields = (
        "co_argcount", "co_posonlyargcount", "co_kwonlyargcount", "co_nlocals",
        "co_stacksize", "co_flags", "co_code", "co_consts", "co_names", "co_varnames",
        "co_name", "co_qualname", "co_cellvars", "co_freevars", "co_exceptiontable",
    )
    return {field: _s4_core_value(getattr(code, field)) for field in fields if hasattr(code, field)}


def _s4_core_value(value, active=None):
    """Canonicalize scalar/container code values and defaults without changing them."""
    if isinstance(value, CodeType):
        return {"code": _s4_core_code_value(value)}
    if value is Ellipsis:
        return {"ellipsis": True}
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return {"float": value.hex()} if math.isfinite(value) else {"float": repr(value)}
    if isinstance(value, complex):
        return {"complex": [_s4_core_value(value.real), _s4_core_value(value.imag)]}
    if isinstance(value, bytes):
        return {"bytes": value.hex()}
    if not isinstance(value, (dict, list, tuple, set, frozenset)):
        raise TypeError(f"Unsupported active core identity value: {type(value).__name__}")
    active = set() if active is None else active
    identity = id(value)
    if identity in active:
        raise ValueError("Cyclic containers cannot define a deterministic core identity.")
    nested = active | {identity}
    if isinstance(value, dict):
        pairs = [[_s4_core_value(key, nested), _s4_core_value(item, nested)] for key, item in value.items()]
        return {"dict": sorted(pairs, key=lambda pair: _s4_core_identity_json(pair[0]))}
    items = [_s4_core_value(item, nested) for item in value]
    if isinstance(value, (set, frozenset)):
        items = sorted(items, key=_s4_core_identity_json)
    return {type(value).__name__: items}


def _s4_core_identity_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def _s4_core_referenced_names(code):
    """Include globals referenced inside comprehensions and other nested code."""
    children = (item for item in code.co_consts if isinstance(item, CodeType))
    return set(code.co_names).union(*( _s4_core_referenced_names(child) for child in children))


def _s4_core_function_value(function):
    closure = tuple(cell.cell_contents for cell in (function.__closure__ or ()))
    return {
        "code": _s4_core_code_value(function.__code__),
        "defaults": _s4_core_value(function.__defaults__),
        "kwdefaults": _s4_core_value(function.__kwdefaults__),
        "closure": _s4_core_value(closure),
    }


def sector4_runtime_core_sha256(entrypoint, namespace, dependency_versions):
    """Fingerprint active core functions, referenced S3 constants and library versions.

    Only Python functions reachable through executable global references and
    referenced ``S4_*`` constants participate. Unrelated configuration, clocks and
    notebook state are ignored. Recursive function graphs terminate through a
    visited-name set; repeated cell executions keep the same identity.
    """
    if not isinstance(entrypoint, FunctionType) or not isinstance(namespace, dict):
        raise TypeError("Provide a Python entrypoint function and its explicit namespace dictionary.")
    if not isinstance(dependency_versions, dict):
        raise TypeError("Dependency versions must be an explicit dictionary.")
    pending = {"entrypoint": entrypoint}
    functions, constants = {}, {}
    while pending:
        name = min(pending)
        function = pending.pop(name)
        if name in functions:
            continue
        functions[name] = _s4_core_function_value(function)
        for reference in sorted(_s4_core_referenced_names(function.__code__)):
            value = namespace.get(reference)
            if isinstance(value, FunctionType) and reference not in functions:
                pending[reference] = value
            elif reference.isupper() and reference in namespace and isinstance(value, (str, bool, int, float, bytes, dict, list, tuple, set, frozenset, type(None))):
                constants[reference] = _s4_core_value(value)
    payload = {
        "schema_version": 1, "functions": functions, "constants": constants,
        "dependency_versions": _s4_core_value(dependency_versions),
    }
    return hashlib.sha256(_s4_core_identity_json(payload).encode("utf-8")).hexdigest()


def sector4_processing_signature(config, lineage):
    versions = {name: importlib.metadata.version(name) for name in ("numpy", "pandas", "pyarrow", "iterative-stratification")}
    core_sha = sector4_runtime_core_sha256(build_sector4_tables, globals(), versions)
    keys = ["seed", "split_strategy", "train_fraction", "validation_fraction", "test_fraction",
            "domain_adaptation_enabled", "domain_adaptation_include_fully_unlabelled_videos",
            "domain_adaptation_include_train_video_unlabelled_frames", "domain_adaptation_sampling_seconds"]
    processing = {key: config[key] for key in keys}
    processing["storage_root"] = config["storage_root"]
    hashes = lineage["input_sha256"]
    payload = {"schema_version": 1, "input_sha256": hashes, "core_source_sha256": core_sha, "processing_config": processing}
    signature = hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()
    lineage["core_source_sha256"] = core_sha
    lineage["dependency_versions"] = versions
    return signature


In [ ]:
PROCESSING_SIGNATURE = sector4_processing_signature(CONFIG, LINEAGE)
CACHED = io_load_cached_sector4(CONFIG, PROCESSING_SIGNATURE)

if CACHED is None:
    TABLES = build_sector4_tables(INPUTS, CONFIG)
    print("Sector 4 transformations computed from verified Sector 3 inputs.")
else:
    TABLES = CACHED["tables"]
    print("Sector 4 transformations reused from verified saved artifacts.")

TABLES["config_inheritance_report"] = LINEAGE["config_conflict_report"].copy()

video_split_manifest = TABLES["video_split_manifest"]
VIDEO_SPLITS = {
    name: set(video_split_manifest.loc[video_split_manifest["split"].eq(name), "video_key"])
    for name in ("train", "validation", "test")
}
SPLIT_LOOKUP = build_split_lookup(VIDEO_SPLITS)
labelled_frame_manifest = TABLES["labelled_frame_manifest_with_split"]
video_manifest = TABLES["video_manifest_with_roles"]
domain_source_videos = TABLES["domain_source_videos"]
domain_adaptation_manifest = TABLES["domain_adaptation_manifest"]
domain_adaptation_summary = TABLES["domain_adaptation_summary"]
VERIFIED_FRAME_LOOKUP = build_verified_frame_lookup(labelled_frame_manifest)


Sector 4 transformations computed from verified Sector 3 inputs.


### 13. Inspect split, class support, and adaptation eligibility

In [ ]:
display(TABLES["split_summary_report"])
display(TABLES["class_split_support_report"])
display(TABLES["video_role_report"])
display(TABLES["domain_eligibility_report"])
display(domain_adaptation_summary)
print("Domain-adaptation frame references:", f"{len(domain_adaptation_manifest):,}")
display(domain_adaptation_manifest.head())


,split,requested_video_fraction,video_count,actual_video_fraction,retained_annotation_count,retained_unique_frame_count,decoded_video_frame_count,partition_validation_passed
0,train,0.70,30,0.697674,34785,34775,1400349,True
1,validation,0.15,5,0.116279,5245,5245,241262,True
2,test,0.15,8,0.186047,7209,7209,337674,True


,finding_class_normalized,split,supporting_video_count,retained_annotation_count,retained_unique_frame_count,total_supporting_video_count,absent_in_split,scarce_video_support,represented_in_all_splits,coverage_status,stratification_guarantees_class_coverage
0,ampulla of vater,train,1,10,10,1,False,True,False,fewer_than_three_supporting_videos,False
1,ampulla of vater,validation,0,0,0,1,True,True,False,fewer_than_three_supporting_videos,False
2,ampulla of vater,test,0,0,0,1,True,True,False,fewer_than_three_supporting_videos,False
3,angiectasia,train,4,719,719,6,False,False,True,represented_in_all_splits,False
4,angiectasia,validation,1,39,39,6,False,False,True,represented_in_all_splits,False
5,angiectasia,test,1,108,108,6,False,False,True,represented_in_all_splits,False
6,blood fresh,train,1,424,424,2,False,True,False,fewer_than_three_supporting_videos,False
7,blood fresh,validation,0,0,0,2,True,True,False,fewer_than_three_supporting_videos,False
8,blood fresh,test,1,22,22,2,False,True,False,fewer_than_three_supporting_videos,False
9,blood hematin,train,1,12,12,1,False,True,False,fewer_than_three_supporting_videos,False


,video_annotation_type,supervised_split,data_role,video_count
0,fully_unlabelled,<NA>,domain_adaptation_only,74
1,partially_labelled,test,supervised_test,8
2,partially_labelled,train,supervised_train_plus_domain_adaptation,30
3,partially_labelled,validation,supervised_validation,5


,video_key,video_annotation_type,supervised_split,data_role,original_annotation_count,retained_annotation_count,excluded_annotation_count,domain_sampling_requested,domain_sampling_eligible,domain_sampling_reason
0,04a78ef00c5245e0,partially_labelled,test,supervised_test,1292,1292,0,False,False,supervised_validation_or_test
1,0531325b64674948,partially_labelled,test,supervised_test,1191,1191,0,False,False,supervised_validation_or_test
2,055bbbec392b4f3a,fully_unlabelled,<NA>,domain_adaptation_only,0,0,0,True,True,fully_unlabelled_source
3,0728084c8da942d9,partially_labelled,train,supervised_train_plus_domain_adaptation,1499,1499,0,True,True,verified_train_source
4,07c1fa15a20a4398,partially_labelled,test,supervised_test,678,678,0,False,False,supervised_validation_or_test
...,...,...,...,...,...,...,...,...,...,...
112,f7acef9e901f405c,fully_unlabelled,<NA>,domain_adaptation_only,0,0,0,True,True,fully_unlabelled_source
113,f89e276571ce4538,fully_unlabelled,<NA>,domain_adaptation_only,0,0,0,True,True,fully_unlabelled_source
114,fb86bc87d3874cd7,partially_labelled,train,supervised_train_plus_domain_adaptation,2085,2085,0,True,True,verified_train_source
115,fc32def0e7194981,partially_labelled,train,supervised_train_plus_domain_adaptation,1380,1380,0,True,True,verified_train_source


,source_type,eligible_video_count,sampled_video_count,frame_reference_count,unlabelled_only_validation_passed
0,fully_unlabelled_video,74,74,185755,True
1,unlabelled_frame_from_train_video,30,30,91041,True


Domain-adaptation frame references: 276,796


,video_key,video_relpath,opencv_frame_index,frame_number,frame_index_offset,source_timestamp_seconds,timestamp_basis,source_type,annotation_status,ground_truth_label,supervised_split,data_role,purpose
0,055bbbec392b4f3a,data/raw/kvasir_capsule/osfstorage/unlabelled_...,0,<NA>,<NA>,0.0,nominal_container_fps_estimate,fully_unlabelled_video,unlabelled,<NA>,<NA>,domain_adaptation_only,visual_domain_adaptation
1,055bbbec392b4f3a,data/raw/kvasir_capsule/osfstorage/unlabelled_...,15,<NA>,<NA>,0.5,nominal_container_fps_estimate,fully_unlabelled_video,unlabelled,<NA>,<NA>,domain_adaptation_only,visual_domain_adaptation
2,055bbbec392b4f3a,data/raw/kvasir_capsule/osfstorage/unlabelled_...,30,<NA>,<NA>,1.0,nominal_container_fps_estimate,fully_unlabelled_video,unlabelled,<NA>,<NA>,domain_adaptation_only,visual_domain_adaptation
3,055bbbec392b4f3a,data/raw/kvasir_capsule/osfstorage/unlabelled_...,45,<NA>,<NA>,1.5,nominal_container_fps_estimate,fully_unlabelled_video,unlabelled,<NA>,<NA>,domain_adaptation_only,visual_domain_adaptation
4,055bbbec392b4f3a,data/raw/kvasir_capsule/osfstorage/unlabelled_...,60,<NA>,<NA>,2.0,nominal_container_fps_estimate,fully_unlabelled_video,unlabelled,<NA>,<NA>,domain_adaptation_only,visual_domain_adaptation


### 14. Persist only changed Sector 4 content

This cell writes the sector's stable results and content-dependent config/report
versions, then commits the checked current catalog last. Upstream sectors remain
inputs.

In [ ]:
SECTOR4_PERSISTED = io_persist_sector4(TABLES, CONFIG, LINEAGE, PROCESSING_SIGNATURE)
print("Sector 4 save status:", SECTOR4_PERSISTED["status"])
print("Saved directory:", SECTOR4_PERSISTED["local_root"])
display(pd.DataFrame(SECTOR4_PERSISTED["artifacts"]))


Sector 4 save status: updated
Saved directory: /content/drive/MyDrive/MMVQA_Clinical/outputs/phase2/sector4


,relative_path,sha256,bytes
0,configs/sector4_config_20261002T011949995661Z....,e48543a7c950cee5df00e189a1338cf3fe80b8e699c31a...,32843
1,reports/class_split_support_report_20261002T01...,4bb408b352183981e8e3b247c9c1f1415ec34eb5be490a...,3803
2,reports/class_split_support_report_20261002T01...,22ba3f5b4a051d167defc28b3d767e7eb1ef7a1adef9e2...,1211
3,reports/config_inheritance_report_20261002T011...,1a1eadc43d0402c00dafcfcb1bd84c19f3bf50c6bc8683...,8505
4,reports/config_inheritance_report_20261002T011...,9948f11ab2d99fad99e160b63858cb6c300b2e7a385517...,681
5,reports/domain_adaptation_summary_20261002T011...,6d5a9c16a9eacb633ec46ca03e3ec013f33bf99ffd6ef1...,201
6,reports/domain_adaptation_summary_20261002T011...,da3b23bba0483dfe61c9b657e99bd8c330cab5f0590f49...,682
7,reports/domain_eligibility_report_20261002T011...,a9185a2ccbd90529c2012332e4cb951a6719fe798593b3...,12850
8,reports/domain_eligibility_report_20261002T011...,cd0ff72e9da908d0c217e3ec93b1e36dd1101c108abd11...,1184
9,reports/executive_report_20261002T011949995661...,2ba1d75d5aec9f7863cf3d29ac296b6b772f630cf2d5a6...,811


### 15. Publish the checked Sector 4 bundle to GitHub

Uses the repository, branch, account, and Colab secret name inherited from Sector 3.
Publication runs from a separate temporary checkout and writes only
`outputs/phase2/sector4`. A normal push is used; concurrent branch changes stop the
publication. With `push=False`, local saved artifacts remain ready for later use.

In [ ]:
if CONFIG["github"]["push"] or CONFIG["github"]["dry_run"]:
    SECTOR4_PUBLICATION = io_publish_sector4(SECTOR4_PERSISTED, CONFIG)
else:
    SECTOR4_PUBLICATION = {"status": "not_requested", "pushed": False}
print(SECTOR4_PUBLICATION)


{'status': 'published', 'branch': 'main', 'export_dir': 'outputs/phase2/sector4', 'count': 26, 'pushed': True, 'dry_run': False, 'changed_paths': ['outputs/phase2/sector4/configs/sector4_config_20261002T011949995661Z.json', 'outputs/phase2/sector4/configs/sector4_state.json', 'outputs/phase2/sector4/reports/class_split_support_report_20261002T011949995661Z.csv', 'outputs/phase2/sector4/reports/class_split_support_report_20261002T011949995661Z_schema.json', 'outputs/phase2/sector4/reports/config_inheritance_report_20261002T011949995661Z.csv', 'outputs/phase2/sector4/reports/config_inheritance_report_20261002T011949995661Z_schema.json', 'outputs/phase2/sector4/reports/domain_adaptation_summary_20261002T011949995661Z.csv', 'outputs/phase2/sector4/reports/domain_adaptation_summary_20261002T011949995661Z_schema.json', 'outputs/phase2/sector4/reports/domain_eligibility_report_20261002T011949995661Z.csv', 'outputs/phase2/sector4/reports/domain_eligibility_report_20261002T011949995661Z_schema.